<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/arc_v10_all4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARC V10 Colab Notebook — All 4 Upgrades

This notebook includes all four requested upgrades:

1. **Role inference**
2. **Better color reasoning**
3. **Visualization**
4. **Batch evaluation over many ARC tasks**

It is designed to be uploaded directly into **Google Colab** from your phone.

In [ ]:
# Setup
import os, json, math, itertools
from collections import deque, Counter

os.makedirs("data/raw", exist_ok=True)
print("Ready.")

Ready.


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'arc_trace_dataset.json', 'arc_compositional_memory_db.json', 'arc_trace_dataset_v2.json', 'arc_memory_db.json', 'submission.json', 'data', 'arc_tasks.json', 'drive', 'arc_compositional_grammar_dataset.json', 'sample_data']


In [ ]:
import json
import os

FILES = [
    "/content/arc_tasks.json",
    "/content/arc_trace_dataset.json",
    "/content/arc_trace_dataset_v2.json"
]

for path in FILES:
    print("=" * 50)
    print("FILE:", path)

    if not os.path.exists(path):
        print("❌ not found")
        continue

    with open(path, "r") as f:
        data = json.load(f)

    print("Type:", type(data))

    if isinstance(data, dict):
        keys = list(data.keys())
        print("Keys:", keys[:5])

        # Check if it looks like ARC
        if "train" in data and "test" in data:
            print("👉 SINGLE ARC TASK FORMAT")

        elif isinstance(data.get(keys[0]), dict):
            sample = data[keys[0]]
            if "train" in sample and "test" in sample:
                print("👉 MULTI ARC TASK FORMAT")

    print()

FILE: /content/arc_tasks.json
Type: <class 'dict'>
Keys: ['demo_task']
👉 MULTI ARC TASK FORMAT

FILE: /content/arc_trace_dataset.json
Type: <class 'list'>

FILE: /content/arc_trace_dataset_v2.json
Type: <class 'list'>



In [ ]:
# ============================================
# FINAL DATA LOAD (CONFIRMED)
# ============================================

import json

DATA_PATH = "/content/arc_tasks.json"

tasks = json.load(open(DATA_PATH))

print("Loaded tasks:", len(tasks))
print("Task IDs:", list(tasks.keys()))

Loaded tasks: 1
Task IDs: ['demo_task']


## Optional: upload your own `arc_tasks.json`

If you already have a task file, run this cell.  
If not, skip it and use the demo task cell below.

In [ ]:
# ============================================
# CONVERT DATASET → SOLVER FORMAT
# ============================================

def convert_to_solver_format(tasks):
    converted = {}

    for task_id, data in tasks.items():
        # Try to extract grids if they exist
        if "train" in data and "test" in data:
            converted[task_id] = data
            continue

        # Otherwise create a dummy structure (for now)
        # (we'll improve this later)
        converted[task_id] = {
            "train": [],
            "test": []
        }

    return converted


solver_tasks = convert_to_solver_format(tasks)

print("✅ Converted tasks:", len(solver_tasks))
print("Example:", list(solver_tasks.keys())[:3])

✅ Converted tasks: 1
Example: ['demo_task']


## Demo task creator

This guarantees the notebook always has a valid file to run.
If you uploaded your own file above, you can skip this cell.

In [ ]:
demo = {
    "demo_task": {
        "train": [
            {
                "input": [[1, 0], [0, 1]],
                "output": [[2, 0], [0, 2]]
            }
        ],
        "test": [
            {
                "input": [[1, 0], [0, 1]]
            }
        ]
    }
}

if not os.path.exists("data/raw/arc_tasks.json"):
    with open("data/raw/arc_tasks.json", "w") as f:
        json.dump(demo, f)
    print("Demo task saved.")
else:
    print("Using existing data/raw/arc_tasks.json")

Using existing data/raw/arc_tasks.json


## Core utilities

In [ ]:
Grid = list[list[int]]

def copy_grid(grid: Grid) -> Grid:
    return [row[:] for row in grid]

def grid_shape(grid: Grid):
    return (len(grid), len(grid[0]) if grid else 0)

def unique_colors(grid: Grid):
    return sorted({v for row in grid for v in row})

def show_grid_ascii(grid: Grid, title="grid"):
    print(f"\n{title} ({len(grid)}x{len(grid[0])})")
    for row in grid:
        print(" ".join(str(v) for v in row))

## Visualization

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

ARC_COLORS = [
    "#000000",
    "#0074D9",
    "#FF4136",
    "#2ECC40",
    "#FFDC00",
    "#AAAAAA",
    "#F012BE",
    "#FF851B",
    "#7FDBFF",
    "#870C25",
]
ARC_CMAP = ListedColormap(ARC_COLORS)

def plot_grid(grid: Grid, title="grid", scale=1.0):
    h, w = len(grid), len(grid[0])
    plt.figure(figsize=(max(3, w*0.5*scale), max(3, h*0.5*scale)))
    plt.imshow(grid, cmap=ARC_CMAP, vmin=0, vmax=9)
    plt.title(title)
    plt.xticks(range(w))
    plt.yticks(range(h))
    plt.grid(True, color="white", linewidth=1)
    plt.show()

def plot_triplet(inp: Grid, out: Grid, pred: Grid, titles=("input", "target", "pred")):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    for ax, g, t in zip(axes, [inp, out, pred], titles):
        ax.imshow(g, cmap=ARC_CMAP, vmin=0, vmax=9)
        ax.set_title(t)
        ax.set_xticks(range(len(g[0])))
        ax.set_yticks(range(len(g)))
        ax.grid(True, color="white", linewidth=1)
    plt.tight_layout()
    plt.show()

## Object extraction + role inference

In [ ]:
def extract_objects(grid: Grid, background_color=0, connectivity=4):
    h, w = len(grid), len(grid[0])
    visited = set()
    objects = []

    if connectivity == 4:
        deltas = [(1,0),(-1,0),(0,1),(0,-1)]
    else:
        deltas = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]

    for r in range(h):
        for c in range(w):
            if (r, c) in visited or grid[r][c] == background_color:
                continue

            color = grid[r][c]
            q = deque([(r, c)])
            visited.add((r, c))
            pixels = []

            while q:
                x, y = q.popleft()
                pixels.append((x, y))
                for dx, dy in deltas:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < h and 0 <= ny < w:
                        if (nx, ny) not in visited and grid[nx][ny] == color:
                            visited.add((nx, ny))
                            q.append((nx, ny))

            rows = [p[0] for p in pixels]
            cols = [p[1] for p in pixels]
            bbox = (min(rows), min(cols), max(rows), max(cols))
            centroid = (sum(rows)/len(rows), sum(cols)/len(cols))
            touches_border = any(pr == 0 or pc == 0 or pr == h-1 or pc == w-1 for pr, pc in pixels)

            objects.append({
                "id": len(objects),
                "color": color,
                "pixels": pixels,
                "area": len(pixels),
                "bbox": bbox,
                "centroid": centroid,
                "touches_border": touches_border,
            })
    return objects

def infer_roles(grid: Grid, background_color=0):
    objs = extract_objects(grid, background_color=background_color)
    if not objs:
        return {"objects": [], "largest": None, "smallest": None, "center": None, "border": [], "minority_color": None}

    largest = max(objs, key=lambda o: (o["area"], -o["id"]))
    smallest = min(objs, key=lambda o: (o["area"], o["id"]))

    h, w = len(grid), len(grid[0])
    center_point = ((h-1)/2.0, (w-1)/2.0)
    center_obj = min(
        objs,
        key=lambda o: (o["centroid"][0]-center_point[0])**2 + (o["centroid"][1]-center_point[1])**2
    )

    border_objs = [o for o in objs if o["touches_border"]]

    color_counts = Counter(v for row in grid for v in row if v != background_color)
    minority_color = min(color_counts, key=color_counts.get) if color_counts else None

    return {
        "objects": objs,
        "largest": largest,
        "smallest": smallest,
        "center": center_obj,
        "border": border_objs,
        "minority_color": minority_color,
    }

## Better color reasoning

In [ ]:
def infer_global_color_map(inp: Grid, out: Grid):
    if grid_shape(inp) != grid_shape(out):
        return None

    mapping = {}
    for r in range(len(inp)):
        for c in range(len(inp[0])):
            src = inp[r][c]
            dst = out[r][c]
            if src in mapping and mapping[src] != dst:
                return None
            mapping[src] = dst
    return mapping

def infer_object_recolor_rule(inp: Grid, out: Grid, background_color=0):
    if grid_shape(inp) != grid_shape(out):
        return None

    roles = infer_roles(inp, background_color)
    if roles["largest"] is None:
        return None

    candidates = [
        ("largest", roles["largest"]),
        ("smallest", roles["smallest"]),
        ("center", roles["center"]),
    ]

    for role_name, obj in candidates:
        if obj is None:
            continue

        changed_colors = set()
        changed_elsewhere = False
        pixel_set = set(obj["pixels"])

        for r in range(len(inp)):
            for c in range(len(inp[0])):
                if inp[r][c] != out[r][c]:
                    if (r, c) in pixel_set:
                        changed_colors.add(out[r][c])
                    else:
                        changed_elsewhere = True

        if not changed_elsewhere and len(changed_colors) == 1:
            return {"role": role_name, "color": list(changed_colors)[0]}
    return None

## DSL operations

In [ ]:
def op_identity(grid: Grid):
    return copy_grid(grid)

def op_color_map(grid: Grid, mapping):
    return [[mapping.get(v, v) for v in row] for row in grid]

def op_rotate90(grid: Grid):
    return [list(row) for row in zip(*grid[::-1])]

def op_rotate180(grid: Grid):
    return [row[::-1] for row in grid[::-1]]

def op_rotate270(grid: Grid):
    return [list(row) for row in zip(*grid)][::-1]

def op_reflect_h(grid: Grid):
    return grid[::-1]

def op_reflect_v(grid: Grid):
    return [row[::-1] for row in grid]

def op_tile_n(grid: Grid, n: int):
    return [row * n for row in grid for _ in range(n)]

def op_row_alt_tile_n(grid: Grid, n: int):
    out = []
    for tile_row in range(n):
        block = grid if tile_row % 2 == 0 else op_reflect_v(grid)
        for row in block:
            row_out = []
            for tile_col in range(n):
                piece = row if tile_col % 2 == 0 else row[::-1]
                row_out.extend(piece)
            out.append(row_out)
    return out

def _render_object_to_bbox(obj, background=0):
    min_r, min_c, max_r, max_c = obj["bbox"]
    out = [[background for _ in range(max_c-min_c+1)] for _ in range(max_r-min_r+1)]
    for r, c in obj["pixels"]:
        out[r-min_r][c-min_c] = obj["color"]
    return out

def op_select_role(grid: Grid, role: str, background_color=0):
    roles = infer_roles(grid, background_color)
    obj = roles.get(role)
    if obj is None:
        return [[background_color]]
    return _render_object_to_bbox(obj, background=background_color)

def op_recolor_role(grid: Grid, role: str, color: int, background_color=0):
    roles = infer_roles(grid, background_color)
    obj = roles.get(role)
    if obj is None:
        return copy_grid(grid)
    out = copy_grid(grid)
    for r, c in obj["pixels"]:
        out[r][c] = color
    return out

def op_keep_color(grid: Grid, color: int, background_color=0):
    h, w = len(grid), len(grid[0])
    out = [[background_color for _ in range(w)] for _ in range(h)]
    for r in range(h):
        for c in range(w):
            if grid[r][c] == color:
                out[r][c] = color
    return out

def op_translate_role(grid: Grid, role: str, dx: int, dy: int, background_color=0):
    roles = infer_roles(grid, background_color)
    obj = roles.get(role)
    if obj is None:
        return copy_grid(grid)

    h, w = len(grid), len(grid[0])
    out = copy_grid(grid)
    for r, c in obj["pixels"]:
        out[r][c] = background_color
    for r, c in obj["pixels"]:
        nr, nc = r + dy, c + dx
        if 0 <= nr < h and 0 <= nc < w:
            out[nr][nc] = obj["color"]
    return out

## Program executor

In [ ]:
def run_program(grid: Grid, program):
    g = copy_grid(grid)
    for op, kwargs in program:
        if op == "identity":
            g = op_identity(g)
        elif op == "color_map":
            g = op_color_map(g, kwargs["mapping"])
        elif op == "rotate90":
            g = op_rotate90(g)
        elif op == "rotate180":
            g = op_rotate180(g)
        elif op == "rotate270":
            g = op_rotate270(g)
        elif op == "reflect_h":
            g = op_reflect_h(g)
        elif op == "reflect_v":
            g = op_reflect_v(g)
        elif op == "tile_n":
            g = op_tile_n(g, kwargs["n"])
        elif op == "row_alt_tile_n":
            g = op_row_alt_tile_n(g, kwargs["n"])
        elif op == "select_role":
            g = op_select_role(g, kwargs["role"])
        elif op == "recolor_role":
            g = op_recolor_role(g, kwargs["role"], kwargs["color"])
        elif op == "keep_color":
            g = op_keep_color(g, kwargs["color"])
        elif op == "translate_role":
            g = op_translate_role(g, kwargs["role"], kwargs["dx"], kwargs["dy"])
        else:
            raise ValueError(f"Unknown op: {op}")
    return g

## Scoring + beam search

In [ ]:
# ============================================
# SCORING + PROGRAM EVAL + BEAM SEARCH
# ============================================

def pixel_score(pred: Grid, target: Grid):
    if grid_shape(pred) != grid_shape(target):
        return 0.0

    total = len(target) * len(target[0])

    correct = sum(
        1 for r in range(len(target)) for c in range(len(target[0]))
        if pred[r][c] == target[r][c]
    )

    return correct / total if total else 0.0


def eval_program_on_task(task, program):
    scores = []
    for pair in task["train"]:
        pred = run_program(pair["input"], program)
        scores.append(pixel_score(pred, pair["output"]))

    return sum(scores)/len(scores) if scores else 0.0, scores


def build_task_ops(task):
    out_colors = sorted({
        v for pair in task["train"]
        for row in pair["output"]
        for v in row if v != 0
    })

    roles = ["largest", "smallest", "center"]

    ops = [
        ("identity", {}),
        ("rotate90", {}),
        ("rotate180", {}),
        ("rotate270", {}),
        ("reflect_h", {}),
        ("reflect_v", {}),
        ("tile_n", {"n": 2}),
        ("tile_n", {"n": 3}),
        ("row_alt_tile_n", {"n": 2}),
        ("row_alt_tile_n", {"n": 3}),
    ]

    for role in roles:
        ops.append(("select_role", {"role": role}))
        ops.append(("translate_role", {"role": role, "dx": 1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": -1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": 1}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": -1}))

        for color in out_colors[:6]:
            ops.append(("recolor_role", {"role": role, "color": color}))

    for color in out_colors[:6]:
        ops.append(("keep_color", {"color": color}))

    return ops


def seed_programs_from_reasoning(task):
    seeds = []
    pair = task["train"][0]

    gmap = infer_global_color_map(pair["input"], pair["output"])
    if gmap is not None:
        seeds.append([("color_map", {"mapping": gmap})])

    obj_rule = infer_object_recolor_rule(pair["input"], pair["output"])
    if obj_rule is not None:
        seeds.append([
            ("recolor_role", {
                "role": obj_rule["role"],
                "color": obj_rule["color"]
            })
        ])

    if not seeds:
        seeds = [[("identity", {})]]

    return seeds


def program_key(program):
    return repr(program)


def beam_search(task, width=12, depth=3):
    ops = build_task_ops(task)
    beam = seed_programs_from_reasoning(task)
    seen = set(program_key(p) for p in beam)

    ranked = []
    best_program = beam[0]
    best_score = -1.0
    best_pair_scores = []

    # initial scoring
    for p in beam:
        sc, pair_scores = eval_program_on_task(task, p)
        ranked.append((sc, -len(p), p, pair_scores))

        if sc > best_score:
            best_score = sc
            best_program = p
            best_pair_scores = pair_scores

    # beam expansion
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = program + [op]
                k = program_key(new_program)

                if k in seen:
                    continue

                seen.add(k)

                sc, pair_scores = eval_program_on_task(task, new_program)

                candidates.append((sc, -len(new_program), new_program, pair_scores))

                if sc > best_score:
                    best_score = sc
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    # final ranking
    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        if item[2] != attempt_1:
            attempt_2 = item[2]
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }

## Load tasks

In [ ]:
with open("data/raw/arc_tasks.json", "r") as f:
    tasks = json.load(f)

print("Loaded task ids:", list(tasks.keys())[:10])
print("Task count:", len(tasks))

Loaded task ids: ['demo_task']
Task count: 1


## Single-task solve + visualization

## Batch evaluation over many tasks

In [ ]:
def batch_evaluate(tasks, width=10, depth=3, limit=None):
    rows = []
    task_ids = list(tasks.keys())
    if limit is not None:
        task_ids = task_ids[:limit]

    for task_id in task_ids:
        task = tasks[task_id]
        result = beam_search(task, width=width, depth=depth)
        rows.append({
            "task_id": task_id,
            "best_score": result["best_score"],
            "program_len": len(result["best_program"]),
            "best_program": result["best_program"],
        })
    return rows

batch_rows = batch_evaluate(tasks, width=10, depth=3, limit=20)
print("Batch done. Example rows:")
for row in batch_rows[:5]:
    print(row)

Batch done. Example rows:
{'task_id': 'demo_task', 'best_score': 1.0, 'program_len': 1, 'best_program': [('color_map', {'mapping': {1: 2, 0: 0}})]}


## Submission builder

This creates a competition-style dictionary with `attempt_1` and `attempt_2`.

In [ ]:
def build_submission(tasks):
    submission = {}
    for task_id, task in tasks.items():
        result = beam_search(task, width=10, depth=3)
        outputs = []
        for test_case in task["test"]:
            test_input = test_case["input"]
            outputs.append({
                "attempt_1": run_program(test_input, result["attempt_1"]),
                "attempt_2": run_program(test_input, result["attempt_2"]),
            })
        submission[task_id] = outputs
    return submission

submission = build_submission(tasks)

with open("submission.json", "w") as f:
    json.dump(submission, f)

print("submission.json written")
first_key = list(submission.keys())[0]
print("Sample:", first_key, submission[first_key])

submission.json written
Sample: demo_task [{'attempt_1': [[2, 0], [0, 2]], 'attempt_2': [[2, 0], [0, 2]]}]


## Download `submission.json`

In [ ]:
try:
    from google.colab import files
    files.download("submission.json")
except Exception as e:
    print("Download not available here:", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================
# SCORING + PROGRAM EVAL + BEAM SEARCH
# ============================================

def pixel_score(pred: Grid, target: Grid):
    if grid_shape(pred) != grid_shape(target):
        return 0.0

    total = len(target) * len(target[0])

    correct = sum(
        1 for r in range(len(target)) for c in range(len(target[0]))
        if pred[r][c] == target[r][c]
    )

    return correct / total if total else 0.0


def eval_program_on_task(task, program):
    scores = []
    for pair in task["train"]:
        pred = run_program(pair["input"], program)
        scores.append(pixel_score(pred, pair["output"]))

    return sum(scores)/len(scores) if scores else 0.0, scores


def build_task_ops(task):
    out_colors = sorted({
        v for pair in task["train"]
        for row in pair["output"]
        for v in row if v != 0
    })

    roles = ["largest", "smallest", "center"]

    ops = [
        ("identity", {}),
        ("rotate90", {}),
        ("rotate180", {}),
        ("rotate270", {}),
        ("reflect_h", {}),
        ("reflect_v", {}),
        ("tile_n", {"n": 2}),
        ("tile_n", {"n": 3}),
        ("row_alt_tile_n", {"n": 2}),
        ("row_alt_tile_n", {"n": 3}),
    ]

    for role in roles:
        ops.append(("select_role", {"role": role}))
        ops.append(("translate_role", {"role": role, "dx": 1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": -1, "dy": 0}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": 1}))
        ops.append(("translate_role", {"role": role, "dx": 0, "dy": -1}))

        for color in out_colors[:6]:
            ops.append(("recolor_role", {"role": role, "color": color}))

    for color in out_colors[:6]:
        ops.append(("keep_color", {"color": color}))

    return ops


def seed_programs_from_reasoning(task):
    seeds = []
    pair = task["train"][0]

    gmap = infer_global_color_map(pair["input"], pair["output"])
    if gmap is not None:
        seeds.append([("color_map", {"mapping": gmap})])

    obj_rule = infer_object_recolor_rule(pair["input"], pair["output"])
    if obj_rule is not None:
        seeds.append([
            ("recolor_role", {
                "role": obj_rule["role"],
                "color": obj_rule["color"]
            })
        ])

    if not seeds:
        seeds = [[("identity", {})]]

    return seeds


def program_key(program):
    return repr(program)


def beam_search(task, width=12, depth=3):
    ops = build_task_ops(task)
    beam = seed_programs_from_reasoning(task)
    seen = set(program_key(p) for p in beam)

    ranked = []
    best_program = beam[0]
    best_score = -1.0
    best_pair_scores = []

    # initial scoring
    for p in beam:
        sc, pair_scores = eval_program_on_task(task, p)
        ranked.append((sc, -len(p), p, pair_scores))

        if sc > best_score:
            best_score = sc
            best_program = p
            best_pair_scores = pair_scores

    # beam expansion
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = program + [op]
                k = program_key(new_program)

                if k in seen:
                    continue

                seen.add(k)

                sc, pair_scores = eval_program_on_task(task, new_program)

                candidates.append((sc, -len(new_program), new_program, pair_scores))

                if sc > best_score:
                    best_score = sc
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    # final ranking
    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        if item[2] != attempt_1:
            attempt_2 = item[2]
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

In [ ]:
# ============================================
# RUN BEAM SEARCH (DEBUG + SAFE)
# ============================================

# --- DEFINE TEST TASK ---
task = {
    "train": [
        {
            "input": [
                [1, 0],
                [0, 1]
            ],
            "output": [
                [2, 0],
                [0, 2]
            ]
        }
    ],
    "test": [
        {
            "input": [
                [1, 0],
                [0, 1]
            ]
        }
    ]
}

print("✅ Task loaded")


# --- VERIFY CRITICAL FUNCTIONS EXIST ---
required_functions = [
    "beam_search",
    "run_program",
    "pixel_score",
]

for fn in required_functions:
    if fn not in globals():
        raise Exception(f"❌ Missing function: {fn}")

print("✅ All required functions found")


# --- RUN BEAM SEARCH ---
try:
    result = beam_search(task, width=10, depth=3)
    print("\n✅ Beam search executed successfully")

except Exception as e:
    print("\n❌ ERROR DURING BEAM SEARCH")
    import traceback
    traceback.print_exc()
    raise


# --- PRINT RESULTS CLEANLY ---
import pprint

print("\n" + "="*60)
print("📊 RESULT SUMMARY")
print("="*60)

pprint.pprint(result)


# --- APPLY BEST PROGRAM TO TEST INPUT ---
try:
    best_program = result["best_program"]
    test_input = task["test"][0]["input"]

    prediction = run_program(test_input, best_program)

    print("\n" + "="*60)
    print("🧠 BEST PROGRAM")
    print("="*60)
    pprint.pprint(best_program)

    print("\n" + "="*60)
    print("🎯 PREDICTION")
    print("="*60)
    for row in prediction:
        print(row)

except Exception as e:
    print("\n❌ ERROR DURING FINAL EXECUTION")
    import traceback
    traceback.print_exc()

✅ Task loaded
✅ All required functions found

✅ Beam search executed successfully

📊 RESULT SUMMARY
{'attempt_1': [('color_map', {'mapping': {0: 0, 1: 2}})],
 'attempt_2': [('color_map', {'mapping': {0: 0, 1: 2}}), ('identity', {})],
 'best_program': [('color_map', {'mapping': {0: 0, 1: 2}})],
 'best_score': 1.0,
 'pair_scores': [1.0],
 'ranked': [(1.0,
             -2,
             [('color_map', {'mapping': {0: 0, 1: 2}}), ('identity', {})],
             [1.0]),
            (1.0,
             -2,
             [('color_map', {'mapping': {0: 0, 1: 2}}), ('rotate180', {})],
             [1.0]),
            (1.0,
             -2,
             [('color_map', {'mapping': {0: 0, 1: 2}}),
              ('recolor_role', {'color': 2, 'role': 'largest'})],
             [1.0]),
            (1.0,
             -2,
             [('color_map', {'mapping': {0: 0, 1: 2}}),
              ('recolor_role', {'color': 2, 'role': 'smallest'})],
             [1.0]),
            (1.0,
             -2,
       

In [ ]:
# ============================================
# 1. OBJECT EXTRACTOR V2
# ============================================

from collections import deque

def get_grid_shape(grid):
    return len(grid), len(grid[0])


def in_bounds(grid, r, c):
    h, w = get_grid_shape(grid)
    return 0 <= r < h and 0 <= c < w


def get_neighbors(r, c, connectivity=4):
    if connectivity == 4:
        return [
            (r - 1, c),
            (r + 1, c),
            (r, c - 1),
            (r, c + 1),
        ]
    elif connectivity == 8:
        return [
            (r - 1, c),
            (r + 1, c),
            (r, c - 1),
            (r, c + 1),
            (r - 1, c - 1),
            (r - 1, c + 1),
            (r + 1, c - 1),
            (r + 1, c + 1),
        ]
    else:
        raise ValueError("connectivity must be 4 or 8")


def compute_bbox(pixels):
    rows = [r for r, c in pixels]
    cols = [c for r, c in pixels]
    return (
        min(rows),
        min(cols),
        max(rows),
        max(cols),
    )


def compute_centroid(pixels):
    if not pixels:
        return (0.0, 0.0)
    row_mean = sum(r for r, c in pixels) / len(pixels)
    col_mean = sum(c for r, c in pixels) / len(pixels)
    return (row_mean, col_mean)


def touches_border(pixels, grid_shape):
    h, w = grid_shape
    for r, c in pixels:
        if r == 0 or c == 0 or r == h - 1 or c == w - 1:
            return True
    return False


def extract_mask_from_pixels(pixels, bbox, color):
    rmin, cmin, rmax, cmax = bbox
    height = rmax - rmin + 1
    width = cmax - cmin + 1

    mask = [[0 for _ in range(width)] for _ in range(height)]
    for r, c in pixels:
        mask[r - rmin][c - cmin] = color

    return mask


def extract_objects(grid, background=0, connectivity=4):
    h, w = get_grid_shape(grid)
    visited = set()
    objects = []

    for r in range(h):
        for c in range(w):
            if (r, c) in visited:
                continue

            color = grid[r][c]

            if color == background:
                continue

            queue = deque()
            queue.append((r, c))
            visited.add((r, c))

            pixels = []

            while queue:
                cr, cc = queue.popleft()
                pixels.append((cr, cc))

                for nr, nc in get_neighbors(cr, cc, connectivity=connectivity):
                    if not in_bounds(grid, nr, nc):
                        continue

                    if (nr, nc) in visited:
                        continue

                    if grid[nr][nc] != color:
                        continue

                    visited.add((nr, nc))
                    queue.append((nr, nc))

            bbox = compute_bbox(pixels)
            centroid = compute_centroid(pixels)
            area = len(pixels)

            rmin, cmin, rmax, cmax = bbox
            height = rmax - rmin + 1
            width = cmax - cmin + 1

            obj = {
                "id": len(objects),
                "color": color,
                "pixels": pixels,
                "area": area,
                "bbox": bbox,
                "centroid": centroid,
                "height": height,
                "width": width,
                "touches_border": touches_border(pixels, (h, w)),
                "mask": extract_mask_from_pixels(pixels, bbox, color),
            }

            objects.append(obj)

    return objects


# ============================================
# 2. ROLE INFERENCE (V2 ONLY)
# ============================================

def infer_object_roles_v2(objects, grid):
    if not objects:
        return {
            "largest": None,
            "smallest": None,
            "topmost": None,
            "bottommost": None,
            "leftmost": None,
            "rightmost": None,
            "center_most": None,
            "border_objects": [],
        }

    h, w = get_grid_shape(grid)
    grid_center = ((h - 1) / 2.0, (w - 1) / 2.0)

    largest = max(objects, key=lambda o: o["area"])
    smallest = min(objects, key=lambda o: o["area"])

    topmost = min(objects, key=lambda o: o["bbox"][0])
    bottommost = max(objects, key=lambda o: o["bbox"][2])
    leftmost = min(objects, key=lambda o: o["bbox"][1])
    rightmost = max(objects, key=lambda o: o["bbox"][3])

    def center_distance(obj):
        cr, cc = obj["centroid"]
        return (cr - grid_center[0]) ** 2 + (cc - grid_center[1]) ** 2

    center_most = min(objects, key=center_distance)

    border_objects = [obj for obj in objects if obj["touches_border"]]

    return {
        "largest": largest,
        "smallest": smallest,
        "topmost": topmost,
        "bottommost": bottommost,
        "leftmost": leftmost,
        "rightmost": rightmost,
        "center_most": center_most,
        "border_objects": border_objects,
    }


# ============================================
# 3. DEBUG PRINTER
# ============================================

def print_objects(objects):
    print("\nDetected Objects:", len(objects))
    print("-" * 60)

    for obj in objects:
        print(f"Object ID: {obj['id']}")
        print(f"  Color: {obj['color']}")
        print(f"  Area: {obj['area']}")
        print(f"  BBox: {obj['bbox']}")
        print(f"  Centroid: {obj['centroid']}")
        print(f"  Width x Height: {obj['width']} x {obj['height']}")
        print(f"  Touches Border: {obj['touches_border']}")
        print(f"  Pixels: {obj['pixels']}")
        print("  Mask:")
        for row in obj["mask"]:
            print("   ", row)
        print("-" * 60)


def print_roles(roles):
    print("\nInferred Roles")
    print("-" * 60)

    for key, value in roles.items():
        if key == "border_objects":
            print(f"{key}: {[obj['id'] for obj in value]}")
        elif value is None:
            print(f"{key}: None")
        else:
            print(f"{key}: Object {value['id']}")


# ============================================
# 4. TEST GRID
# ============================================

test_grid = [
    [0, 1, 1, 0, 0, 2],
    [0, 1, 0, 0, 2, 2],
    [0, 0, 0, 0, 0, 0],
    [3, 3, 0, 4, 4, 4],
    [3, 0, 0, 0, 4, 0],
]

print("Test Grid:")
for row in test_grid:
    print(row)


# ============================================
# 5. RUN EXTRACTION (4-CONNECTIVITY)
# ============================================

objects_4 = extract_objects(test_grid, background=0, connectivity=4)
roles_4 = infer_object_roles_v2(objects_4, test_grid)

print("\n=== OBJECTS WITH 4-CONNECTIVITY ===")
print_objects(objects_4)
print_roles(roles_4)


# ============================================
# 6. OPTIONAL 8-CONNECTIVITY TEST (FIXED)
# ============================================

objects_8 = extract_objects(test_grid, background=0, connectivity=8)
roles_8 = infer_object_roles_v2(objects_8, test_grid)  # ✅ FIXED LINE

print("\n=== OBJECTS WITH 8-CONNECTIVITY ===")
print_objects(objects_8)
print_roles(roles_8)

Test Grid:
[0, 1, 1, 0, 0, 2]
[0, 1, 0, 0, 2, 2]
[0, 0, 0, 0, 0, 0]
[3, 3, 0, 4, 4, 4]
[3, 0, 0, 0, 4, 0]

=== OBJECTS WITH 4-CONNECTIVITY ===

Detected Objects: 4
------------------------------------------------------------
Object ID: 0
  Color: 1
  Area: 3
  BBox: (0, 1, 1, 2)
  Centroid: (0.3333333333333333, 1.3333333333333333)
  Width x Height: 2 x 2
  Touches Border: True
  Pixels: [(0, 1), (1, 1), (0, 2)]
  Mask:
    [1, 1]
    [1, 0]
------------------------------------------------------------
Object ID: 1
  Color: 2
  Area: 3
  BBox: (0, 4, 1, 5)
  Centroid: (0.6666666666666666, 4.666666666666667)
  Width x Height: 2 x 2
  Touches Border: True
  Pixels: [(0, 5), (1, 5), (1, 4)]
  Mask:
    [0, 2]
    [2, 2]
------------------------------------------------------------
Object ID: 2
  Color: 3
  Area: 3
  BBox: (3, 0, 4, 1)
  Centroid: (3.3333333333333335, 0.3333333333333333)
  Width x Height: 2 x 2
  Touches Border: True
  Pixels: [(3, 0), (4, 0), (3, 1)]
  Mask:
    [3, 3]
    [

In [ ]:
# ============================================
# 7. ROLE OPS UPGRADE (OBJECT EXTRACTOR V2)
# ============================================

def get_roles_v2(grid, background_color=0, connectivity=4):
    objects = extract_objects(grid, background=background_color, connectivity=connectivity)
    roles = infer_object_roles_v2(objects, grid)
    roles["objects"] = objects
    return roles


def render_object_mask(obj, background_color=0):
    return [row[:] for row in obj["mask"]]


def op_select_role_v2(grid, role, background_color=0, connectivity=4):
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [[background_color]]

    obj = roles[role]
    if obj is None:
        return [[background_color]]

    return render_object_mask(obj, background_color=background_color)


def op_recolor_role_v2(grid, role, new_color, background_color=0, connectivity=4):
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [row[:] for row in grid]

    obj = roles[role]
    if obj is None:
        return [row[:] for row in grid]

    out = [row[:] for row in grid]

    for r, c in obj["pixels"]:
        out[r][c] = new_color

    return out


def op_translate_role_v2(grid, role, dx, dy, background_color=0, connectivity=4):
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [row[:] for row in grid]

    obj = roles[role]
    if obj is None:
        return [row[:] for row in grid]

    h, w = len(grid), len(grid[0])
    out = [row[:] for row in grid]

    # erase original object
    for r, c in obj["pixels"]:
        out[r][c] = background_color

    # draw translated object
    for r, c in obj["pixels"]:
        nr = r + dy
        nc = c + dx

        if 0 <= nr < h and 0 <= nc < w:
            out[nr][nc] = obj["color"]

    return out


# quick sanity test
print("=== SELECT largest ===")
selected = op_select_role_v2(test_grid, "largest")
for row in selected:
    print(row)

print("\n=== RECOLOR largest -> 9 ===")
recolored = op_recolor_role_v2(test_grid, "largest", 9)
for row in recolored:
    print(row)

print("\n=== TRANSLATE largest by dx=-1 dy=0 ===")
translated = op_translate_role_v2(test_grid, "largest", -1, 0)
for row in translated:
    print(row)

=== SELECT largest ===
[4, 4, 4]
[0, 4, 0]

=== RECOLOR largest -> 9 ===
[0, 1, 1, 0, 0, 2]
[0, 1, 0, 0, 2, 2]
[0, 0, 0, 0, 0, 0]
[3, 3, 0, 9, 9, 9]
[3, 0, 0, 0, 9, 0]

=== TRANSLATE largest by dx=-1 dy=0 ===
[0, 1, 1, 0, 0, 2]
[0, 1, 0, 0, 2, 2]
[0, 0, 0, 0, 0, 0]
[3, 3, 4, 4, 4, 0]
[3, 0, 0, 4, 0, 0]


In [ ]:
roles = [
    "largest",
    "smallest",
    "center_most",
    "topmost",
    "bottommost",
    "leftmost",
    "rightmost",
]

In [ ]:
# ============================================
# 8. RUN PROGRAM (V2 OBJECT-AWARE VERSION)
# ============================================

def run_program(grid, program):
    current = [row[:] for row in grid]

    for op_name, params in program:

        # --- BASIC OPS ---
        if op_name == "identity":
            continue

        elif op_name == "rotate90":
            current = list(zip(*current[::-1]))
            current = [list(row) for row in current]

        elif op_name == "rotate180":
            current = [row[::-1] for row in current[::-1]]

        elif op_name == "rotate270":
            current = list(zip(*current))[::-1]
            current = [list(row) for row in current]

        elif op_name == "reflect_h":
            current = current[::-1]

        elif op_name == "reflect_v":
            current = [row[::-1] for row in current]

        # --- TILING ---
        elif op_name == "tile_n":
            n = params.get("n", 2)
            current = [row * n for row in current] * n

        elif op_name == "row_alt_tile_n":
            n = params.get("n", 2)
            new_grid = []
            for i in range(len(current)):
                row = current[i]
                new_row = []
                for j in range(n):
                    if j % 2 == 0:
                        new_row += row
                    else:
                        new_row += row[::-1]
                new_grid.append(new_row)
            current = new_grid * n

        # --- COLOR MAP ---
        elif op_name == "color_map":
            mapping = params.get("mapping", {})
            current = [
                [mapping.get(cell, cell) for cell in row]
                for row in current
            ]

        elif op_name == "keep_color":
            color = params.get("color")
            current = [
                [cell if cell == color else 0 for cell in row]
                for row in current
            ]

        # ============================================
        # 🔥 OBJECT-LEVEL OPS (V2)
        # ============================================

        elif op_name == "select_role":
            current = op_select_role_v2(
                current,
                role=params.get("role")
            )

        elif op_name == "recolor_role":
            current = op_recolor_role_v2(
                current,
                role=params.get("role"),
                new_color=params.get("color")
            )

        elif op_name == "translate_role":
            current = op_translate_role_v2(
                current,
                role=params.get("role"),
                dx=params.get("dx", 0),
                dy=params.get("dy", 0)
            )

        else:
            raise ValueError(f"Unknown operation: {op_name}")

    return current

In [ ]:
# ============================================
# 9. FULL DEBUG OUTPUT (ALL SYSTEM STATE)
# ============================================

import pprint

def debug_full_state(task, result):
    print("\n" + "="*70)
    print("🧠 FULL SYSTEM DEBUG OUTPUT")
    print("="*70)

    # --- INPUT ---
    print("\n📥 INPUT GRID:")
    for row in task["train"][0]["input"]:
        print(row)

    # --- OUTPUT ---
    print("\n📤 TARGET OUTPUT:")
    for row in task["train"][0]["output"]:
        print(row)

    # --- OBJECTS ---
    print("\n🔍 OBJECT EXTRACTION:")
    objects = extract_objects(task["train"][0]["input"])
    print_objects(objects)

    # --- ROLES ---
    print("\n🎭 ROLE ASSIGNMENT:")
    roles = infer_object_roles_v2(objects, task["train"][0]["input"])
    print_roles(roles)

    # --- PROGRAMS ---
    print("\n🧠 BEST PROGRAM:")
    pprint.pprint(result["best_program"])

    print("\n🎯 ATTEMPT 1:")
    pprint.pprint(result["attempt_1"])

    print("\n🎯 ATTEMPT 2:")
    pprint.pprint(result["attempt_2"])

    # --- SCORES ---
    print("\n📊 SCORE:")
    print("Best Score:", result["best_score"])
    print("Pair Scores:", result["pair_scores"])

    # --- PREDICTION ---
    print("\n🚀 PREDICTION:")
    pred = run_program(task["test"][0]["input"], result["best_program"])
    for row in pred:
        print(row)

    # --- TOP PROGRAMS ---
    print("\n🔥 TOP CANDIDATES:")
    for sc, _, prog, pair_scores in result["ranked"][:5]:
        print(f"\nScore: {round(sc,3)} | PairScores: {pair_scores}")
        pprint.pprint(prog)

    print("\n" + "="*70)


# ============================================
# RUN DEBUG
# ============================================

debug_full_state(task, result)


🧠 FULL SYSTEM DEBUG OUTPUT

📥 INPUT GRID:
[1, 0]
[0, 1]

📤 TARGET OUTPUT:
[2, 0]
[0, 2]

🔍 OBJECT EXTRACTION:

Detected Objects: 2
------------------------------------------------------------
Object ID: 0
  Color: 1
  Area: 1
  BBox: (0, 0, 0, 0)
  Centroid: (0.0, 0.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(0, 0)]
  Mask:
    [1]
------------------------------------------------------------
Object ID: 1
  Color: 1
  Area: 1
  BBox: (1, 1, 1, 1)
  Centroid: (1.0, 1.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(1, 1)]
  Mask:
    [1]
------------------------------------------------------------

🎭 ROLE ASSIGNMENT:

Inferred Roles
------------------------------------------------------------
largest: Object 0
smallest: Object 0
topmost: Object 0
bottommost: Object 1
leftmost: Object 0
rightmost: Object 1
center_most: Object 0
border_objects: [0, 1]

🧠 BEST PROGRAM:
[('color_map', {'mapping': {0: 0, 1: 2}})]

🎯 ATTEMPT 1:
[('color_map', {'mapping': {0: 0, 

In [ ]:
# ============================================
# 10. ROLE DISAMBIGUATION (TIE BREAKER)
# ============================================

def break_ties(objects, key_fn, secondary_fn=None):
    sorted_objs = sorted(objects, key=key_fn)

    # detect tie
    best_val = key_fn(sorted_objs[0])
    tied = [o for o in sorted_objs if key_fn(o) == best_val]

    if len(tied) == 1:
        return sorted_objs[0]

    # 🔥 tie-breaker using centroid (top-left priority)
    if secondary_fn is None:
        return min(tied, key=lambda o: (o["centroid"][0], o["centroid"][1]))
    else:
        return min(tied, key=secondary_fn)


def infer_object_roles_v2(objects, grid):
    if not objects:
        return {}

    h, w = len(grid), len(grid[0])
    grid_center = ((h - 1) / 2.0, (w - 1) / 2.0)

    largest = break_ties(objects, lambda o: -o["area"])
    smallest = break_ties(objects, lambda o: o["area"])

    topmost = break_ties(objects, lambda o: o["bbox"][0])
    bottommost = break_ties(objects, lambda o: -o["bbox"][2])
    leftmost = break_ties(objects, lambda o: o["bbox"][1])
    rightmost = break_ties(objects, lambda o: -o["bbox"][3])

    def center_dist(o):
        cr, cc = o["centroid"]
        return (cr - grid_center[0])**2 + (cc - grid_center[1])**2

    center_most = break_ties(objects, center_dist)

    border_objects = [o for o in objects if o["touches_border"]]

    return {
        "largest": largest,
        "smallest": smallest,
        "topmost": topmost,
        "bottommost": bottommost,
        "leftmost": leftmost,
        "rightmost": rightmost,
        "center_most": center_most,
        "border_objects": border_objects,
    }

In [ ]:
# ============================================
# 10. ROLE DISAMBIGUATION (TIE BREAKER)
# ============================================

def break_ties(objects, key_fn, secondary_fn=None):
    sorted_objs = sorted(objects, key=key_fn)

    # detect tie
    best_val = key_fn(sorted_objs[0])
    tied = [o for o in sorted_objs if key_fn(o) == best_val]

    if len(tied) == 1:
        return sorted_objs[0]

    # 🔥 tie-breaker using centroid (top-left priority)
    if secondary_fn is None:
        return min(tied, key=lambda o: (o["centroid"][0], o["centroid"][1]))
    else:
        return min(tied, key=secondary_fn)


def infer_object_roles_v2(objects, grid):
    if not objects:
        return {}

    h, w = len(grid), len(grid[0])
    grid_center = ((h - 1) / 2.0, (w - 1) / 2.0)

    largest = break_ties(objects, lambda o: -o["area"])
    smallest = break_ties(objects, lambda o: o["area"])

    topmost = break_ties(objects, lambda o: o["bbox"][0])
    bottommost = break_ties(objects, lambda o: -o["bbox"][2])
  # ============================================
# 10. ROLE DISAMBIGUATION V2 (COMPLEMENTARY TIES)
# ============================================

def break_ties_min(objects, key_fn):
    sorted_objs = sorted(objects, key=key_fn)
    best_val = key_fn(sorted_objs[0])
    tied = [o for o in sorted_objs if key_fn(o) == best_val]

    if len(tied) == 1:
        return tied[0]

    # top-left priority
    return min(tied, key=lambda o: (o["centroid"][0], o["centroid"][1]))


def break_ties_max(objects, key_fn):
    sorted_objs = sorted(objects, key=key_fn)
    best_val = key_fn(sorted_objs[0])
    tied = [o for o in sorted_objs if key_fn(o) == best_val]

    if len(tied) == 1:
        return tied[0]

    # bottom-right priority
    return max(tied, key=lambda o: (o["centroid"][0], o["centroid"][1]))


def infer_object_roles_v2(objects, grid):
    if not objects:
        return {}

    h, w = len(grid), len(grid[0])
    grid_center = ((h - 1) / 2.0, (w - 1) / 2.0)

    # area ties split intentionally
    largest = break_ties_min(objects, lambda o: -o["area"])
    smallest = break_ties_max(objects, lambda o: o["area"])

    topmost = break_ties_min(objects, lambda o: o["bbox"][0])
    bottommost = break_ties_max(objects, lambda o: -o["bbox"][2])

    leftmost = break_ties_min(objects, lambda o: o["bbox"][1])
    rightmost = break_ties_max(objects, lambda o: -o["bbox"][3])

    def center_dist(o):
        cr, cc = o["centroid"]
        return (cr - grid_center[0]) ** 2 + (cc - grid_center[1]) ** 2

    center_most = break_ties_min(objects, center_dist)

    border_objects = [o for o in objects if o["touches_border"]]

    return {
        "largest": largest,
        "smallest": smallest,
        "topmost": topmost,
        "bottommost": bottommost,
        "leftmost": leftmost,
        "rightmost": rightmost,
        "center_most": center_most,
        "border_objects": border_objects,
    }

In [ ]:
# ============================================
# FIX ROLE SYSTEM + REMOVE OLD BUG
# ============================================

# --- SAFE WRAPPER (ensures correct argument name everywhere) ---
def extract_objects_safe(grid, background_color=0, connectivity=4):
    return extract_objects(grid, background=background_color, connectivity=connectivity)


# --- REPLACE ANY OLD infer_roles FUNCTION ---
def infer_roles(grid, background_color=0):
    # FORCE use of new system
    objects = extract_objects_safe(grid, background_color=background_color)
    roles = infer_object_roles_v2(objects, grid)
    roles["objects"] = objects
    return roles


# --- PATCH OLD CALL SITES (if anything still uses old name) ---
# This ensures compatibility with anything in your notebook
def get_roles(grid, background_color=0):
    return infer_roles(grid, background_color=background_color)


# ============================================
# SANITY CHECK
# ============================================

print("Running role system test...")

test_grid_local = [
    [1, 0],
    [0, 1]
]

roles = infer_roles(test_grid_local)

print("\nRoles:")
for k, v in roles.items():
    if k == "objects":
        print("objects:", [o["id"] for o in v])
    elif k == "border_objects":
        print(k, ":", [o["id"] for o in v])
    elif v is None:
        print(k, ": None")
    else:
        print(k, ":", v["id"])

print("\nFix applied successfully.")

Running role system test...

Roles:
largest : 0
smallest : 1
topmost : 0
bottommost : 1
leftmost : 0
rightmost : 1
center_most : 0
border_objects : [0, 1]
objects: [0, 1]

Fix applied successfully.


In [ ]:
# ============================================
# FIX: ROLE KEY COMPATIBILITY (center -> center_most)
# ============================================

def safe_get_role(roles, key):
    # handle legacy keys
    if key == "center":
        return roles.get("center_most", None)
    return roles.get(key, None)


def infer_object_recolor_rule(inp, out, background_color=0):
    roles = infer_roles(inp, background_color=background_color)

    candidates = [
        ("largest", safe_get_role(roles, "largest")),
        ("smallest", safe_get_role(roles, "smallest")),
        ("center", safe_get_role(roles, "center")),  # FIXED
    ]

    for role_name, obj in candidates:
        if obj is None:
            continue

        # collect colors from input vs output at object pixels
        colors = set()
        for r, c in obj["pixels"]:
            colors.add(out[r][c])

        # if exactly one new color → valid rule
        if len(colors) == 1:
            new_color = list(colors)[0]
            if new_color != inp[obj["pixels"][0][0]][obj["pixels"][0][1]]:
                return {
                    "role": role_name,
                    "color": new_color
                }

    return None


print("Role compatibility fix applied.")

Role compatibility fix applied.


In [ ]:
# ============================================
# 12. MULTI-OBJECT ROLE SYSTEM (GROUP ROLES)
# ============================================

def infer_group_roles(objects, grid):
    if not objects:
        return {}

    # --- area grouping ---
    min_area = min(o["area"] for o in objects)
    max_area = max(o["area"] for o in objects)

    smallest_group = [o for o in objects if o["area"] == min_area]
    largest_group = [o for o in objects if o["area"] == max_area]

    # --- border grouping ---
    border_group = [o for o in objects if o["touches_border"]]

    # --- color grouping ---
    color_groups = {}
    for o in objects:
        color_groups.setdefault(o["color"], []).append(o)

    return {
        "smallest_group": smallest_group,
        "largest_group": largest_group,
        "border_group": border_group,
        "color_groups": color_groups,
    }


# ============================================
# GROUP OPERATIONS
# ============================================

def op_recolor_group(grid, role, new_color):
    objects = extract_objects(grid)
    groups = infer_group_roles(objects, grid)

    if role not in groups:
        return [row[:] for row in grid]

    group = groups[role]
    out = [row[:] for row in grid]

    for obj in group:
        for r, c in obj["pixels"]:
            out[r][c] = new_color

    return out


def op_select_group(grid, role):
    objects = extract_objects(grid)
    groups = infer_group_roles(objects, grid)

    if role not in groups:
        return [[0]]

    group = groups[role]

    # build bounding box over all objects
    all_pixels = [p for obj in group for p in obj["pixels"]]

    rmin = min(r for r, c in all_pixels)
    cmin = min(c for r, c in all_pixels)
    rmax = max(r for r, c in all_pixels)
    cmax = max(c for r, c in all_pixels)

    h = rmax - rmin + 1
    w = cmax - cmin + 1

    out = [[0 for _ in range(w)] for _ in range(h)]

    for obj in group:
        for r, c in obj["pixels"]:
            out[r - rmin][c - cmin] = obj["color"]

    return out


# ============================================
# EXTEND run_program FOR GROUP OPS
# ============================================

def run_program(grid, program):
    current = [row[:] for row in grid]

    for op_name, params in program:

        # BASIC
        if op_name == "identity":
            continue

        elif op_name == "color_map":
            mapping = params.get("mapping", {})
            current = [[mapping.get(v, v) for v in row] for row in current]

        elif op_name == "keep_color":
            color = params.get("color")
            current = [[v if v == color else 0 for v in row] for row in current]

        # OBJECT OPS
        elif op_name == "recolor_role":
            current = op_recolor_role_v2(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "translate_role":
            current = op_translate_role_v2(
                current,
                role=params.get("role"),
                dx=params.get("dx", 0),
                dy=params.get("dy", 0),
            )

        elif op_name == "select_role":
            current = op_select_role_v2(
                current,
                role=params.get("role"),
            )

        # ============================================
        # 🔥 NEW GROUP OPS
        # ============================================

        elif op_name == "recolor_group":
            current = op_recolor_group(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "select_group":
            current = op_select_group(
                current,
                role=params.get("role"),
            )

        else:
            pass

    return current


# ============================================
# EXTEND OP GENERATOR
# ============================================

def extend_ops_with_groups(ops):
    group_roles = [
        "smallest_group",
        "largest_group",
        "border_group",
    ]

    for role in group_roles:
        ops.append(("recolor_group", {"role": role, "color": 2}))
        ops.append(("select_group", {"role": role}))

    return ops


print("Multi-object system loaded.")

Multi-object system loaded.


In [ ]:
# ============================================
# 12. MULTI-OBJECT ROLE SYSTEM (GROUP ROLES)
# ============================================

def infer_group_roles(objects, grid):
    if not objects:
        return {}

    # --- area grouping ---
    min_area = min(o["area"] for o in objects)
    max_area = max(o["area"] for o in objects)

    smallest_group = [o for o in objects if o["area"] == min_area]
    largest_group = [o for o in objects if o["area"] == max_area]

    # --- border grouping ---
    border_group = [o for o in objects if o["touches_border"]]

    # --- color grouping ---
    color_groups = {}
    for o in objects:
        color_groups.setdefault(o["color"], []).append(o)

    return {
        "smallest_group": smallest_group,
        "largest_group": largest_group,
        "border_group": border_group,
        "color_groups": color_groups,
    }


# ============================================
# GROUP OPERATIONS
# ============================================

def op_recolor_group(grid, role, new_color):
    objects = extract_objects(grid)
    groups = infer_group_roles(objects, grid)

    if role not in groups:
        return [row[:] for row in grid]

    group = groups[role]
    out = [row[:] for row in grid]

    for obj in group:
        for r, c in obj["pixels"]:
            out[r][c] = new_color

    return out


def op_select_group(grid, role):
    objects = extract_objects(grid)
    groups = infer_group_roles(objects, grid)

    if role not in groups:
        return [[0]]

    group = groups[role]

    # build bounding box over all objects
    all_pixels = [p for obj in group for p in obj["pixels"]]

    rmin = min(r for r, c in all_pixels)
    cmin = min(c for r, c in all_pixels)
    rmax = max(r for r, c in all_pixels)
    cmax = max(c for r, c in all_pixels)

    h = rmax - rmin + 1
    w = cmax - cmin + 1

    out = [[0 for _ in range(w)] for _ in range(h)]

    for obj in group:
        for r, c in obj["pixels"]:
            out[r - rmin][c - cmin] = obj["color"]

    return out


# ============================================
# EXTEND run_program FOR GROUP OPS
# ============================================

def run_program(grid, program):
    current = [row[:] for row in grid]

    for op_name, params in program:

        # BASIC
        if op_name == "identity":
            continue

        elif op_name == "color_map":
            mapping = params.get("mapping", {})
            current = [[mapping.get(v, v) for v in row] for row in current]

        elif op_name == "keep_color":
            color = params.get("color")
            current = [[v if v == color else 0 for v in row] for row in current]

        # OBJECT OPS
        elif op_name == "recolor_role":
            current = op_recolor_role_v2(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "translate_role":
            current = op_translate_role_v2(
                current,
                role=params.get("role"),
                dx=params.get("dx", 0),
                dy=params.get("dy", 0),
            )

        elif op_name == "select_role":
            current = op_select_role_v2(
                current,
                role=params.get("role"),
            )

        # ============================================
        # 🔥 NEW GROUP OPS
        # ============================================

        elif op_name == "recolor_group":
            current = op_recolor_group(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "select_group":
            current = op_select_group(
                current,
                role=params.get("role"),
            )

        else:
            pass

    return current


# ============================================
# EXTEND OP GENERATOR
# ============================================

def extend_ops_with_groups(ops):
    group_roles = [
        "smallest_group",
        "largest_group",
        "border_group",
    ]

    for role in group_roles:
        ops.append(("recolor_group", {"role": role, "color": 2}))
        ops.append(("select_group", {"role": role}))

    return ops


print("Multi-object system loaded.")

Multi-object system loaded.


In [ ]:
# ============================================
# PROGRAM SIMPLICITY + DEDUP FILTER
# ============================================

def normalize_program(program):
    # remove identity ops
    return [op for op in program if op[0] != "identity"]


def program_complexity(program):
    # shorter = better
    return len(normalize_program(program))


def program_signature(program):
    # canonical representation for dedup
    return tuple(normalize_program(program))


def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)

    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []

    # ----------------------------------------
    # INITIAL EVAL
    # ----------------------------------------
    for p in beam:
        p = normalize_program(p)
        sig = program_signature(p)

        if sig in seen:
            continue
        seen.add(sig)

        sc, pair_scores = eval_program_on_task(task, p)

        ranked.append((sc, -program_complexity(p), p, pair_scores))

        if sc > best_score:
            best_score = sc
            best_program = p
            best_pair_scores = pair_scores

    # ----------------------------------------
    # SEARCH LOOP
    # ----------------------------------------
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = normalize_program(program + [op])
                sig = program_signature(new_program)

                if sig in seen:
                    continue
                seen.add(sig)

                sc, pair_scores = eval_program_on_task(task, new_program)

                candidates.append(
                    (sc, -program_complexity(new_program), new_program, pair_scores)
                )

                if sc > best_score:
                    best_score = sc
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    # ----------------------------------------
    # FINAL SELECTION
    # ----------------------------------------
    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        if item[2] != attempt_1:
            attempt_2 = item[2]
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }


print("Beam search upgraded with simplicity + dedup.")

Beam search upgraded with simplicity + dedup.


In [ ]:
# ============================================
# FIX: HASHABLE PROGRAM SIGNATURE
# ============================================

def normalize_program(program):
    return [op for op in program if op[0] != "identity"]


def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    elif isinstance(obj, list):
        return tuple(make_hashable(x) for x in obj)
    else:
        return obj


def program_signature(program):
    norm = normalize_program(program)
    return tuple(
        (op_name, make_hashable(params))
        for op_name, params in norm
    )


def program_complexity(program):
    return len(normalize_program(program))


# ============================================
# UPDATED BEAM SEARCH (SAFE)
# ============================================

def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)

    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []

    # ----------------------------------------
    # INITIAL EVAL
    # ----------------------------------------
    for p in beam:
        p = normalize_program(p)
        sig = program_signature(p)

        if sig in seen:
            continue
        seen.add(sig)

        sc, pair_scores = eval_program_on_task(task, p)

        ranked.append((sc, -program_complexity(p), p, pair_scores))

        if sc > best_score:
            best_score = sc
            best_program = p
            best_pair_scores = pair_scores

    # ----------------------------------------
    # SEARCH LOOP
    # ----------------------------------------
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = normalize_program(program + [op])
                sig = program_signature(new_program)

                if sig in seen:
                    continue
                seen.add(sig)

                sc, pair_scores = eval_program_on_task(task, new_program)

                candidates.append(
                    (sc, -program_complexity(new_program), new_program, pair_scores)
                )

                if sc > best_score:
                    best_score = sc
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    # ----------------------------------------
    # FINAL SELECTION
    # ----------------------------------------
    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        if item[2] != attempt_1:
            attempt_2 = item[2]
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }


print("Beam search fully fixed (hashable + dedup + simplicity).")

Beam search fully fixed (hashable + dedup + simplicity).


In [ ]:
# ============================================
# 13. SYMMETRY DETECTOR + EQUIVALENCE PENALTY
# ============================================

def rotate90_grid(grid):
    return [list(row) for row in zip(*grid[::-1])]

def rotate180_grid(grid):
    return [row[::-1] for row in grid[::-1]]

def rotate270_grid(grid):
    return [list(row) for row in zip(*grid)][::-1]

def reflect_h_grid(grid):
    return grid[::-1]

def reflect_v_grid(grid):
    return [row[::-1] for row in grid]


def same_grid(a, b):
    if len(a) != len(b):
        return False
    if len(a[0]) != len(b[0]):
        return False
    for r in range(len(a)):
        for c in range(len(a[0])):
            if a[r][c] != b[r][c]:
                return False
    return True


def detect_grid_symmetries(grid):
    syms = {
        "rotate90": same_grid(grid, rotate90_grid(grid)),
        "rotate180": same_grid(grid, rotate180_grid(grid)),
        "rotate270": same_grid(grid, rotate270_grid(grid)),
        "reflect_h": same_grid(grid, reflect_h_grid(grid)),
        "reflect_v": same_grid(grid, reflect_v_grid(grid)),
    }
    return syms


def symmetry_penalty(program, task):
    train_input = task["train"][0]["input"]
    syms = detect_grid_symmetries(train_input)

    penalty = 0.0

    for op_name, params in program:
        if op_name in syms and syms[op_name]:
            penalty += 0.02

    return penalty


def normalize_program(program):
    return [op for op in program if op[0] != "identity"]


def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    elif isinstance(obj, list):
        return tuple(make_hashable(x) for x in obj)
    else:
        return obj


def program_signature(program):
    norm = normalize_program(program)
    return tuple((op_name, make_hashable(params)) for op_name, params in norm)


def program_complexity(program):
    return len(normalize_program(program))


def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)

    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []

    # initial eval
    for p in beam:
        p = normalize_program(p)
        sig = program_signature(p)

        if sig in seen:
            continue
        seen.add(sig)

        raw_score, pair_scores = eval_program_on_task(task, p)
        adjusted_score = raw_score - symmetry_penalty(p, task)

        ranked.append((adjusted_score, -program_complexity(p), p, pair_scores))

        if raw_score > best_score:
            best_score = raw_score
            best_program = p
            best_pair_scores = pair_scores

    # search loop
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = normalize_program(program + [op])
                sig = program_signature(new_program)

                if sig in seen:
                    continue
                seen.add(sig)

                raw_score, pair_scores = eval_program_on_task(task, new_program)

                penalty = symmetry_penalty(new_program, task)
                adjusted_score = raw_score - penalty

                candidates.append(
                    (adjusted_score, -program_complexity(new_program), new_program, pair_scores)
                )

                if raw_score > best_score:
                    best_score = raw_score
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        candidate = item[2]
        if candidate != attempt_1:
            attempt_2 = candidate
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }


print("Symmetry-aware beam search loaded.")

Symmetry-aware beam search loaded.


In [ ]:
# ============================================
# 13. SYMMETRY DETECTOR + EQUIVALENCE PENALTY
# ============================================

def rotate90_grid(grid):
    return [list(row) for row in zip(*grid[::-1])]

def rotate180_grid(grid):
    return [row[::-1] for row in grid[::-1]]

def rotate270_grid(grid):
    return [list(row) for row in zip(*grid)][::-1]

def reflect_h_grid(grid):
    return grid[::-1]

def reflect_v_grid(grid):
    return [row[::-1] for row in grid]


def same_grid(a, b):
    if len(a) != len(b):
        return False
    if len(a[0]) != len(b[0]):
        return False
    for r in range(len(a)):
        for c in range(len(a[0])):
            if a[r][c] != b[r][c]:
                return False
    return True


def detect_grid_symmetries(grid):
    syms = {
        "rotate90": same_grid(grid, rotate90_grid(grid)),
        "rotate180": same_grid(grid, rotate180_grid(grid)),
        "rotate270": same_grid(grid, rotate270_grid(grid)),
        "reflect_h": same_grid(grid, reflect_h_grid(grid)),
        "reflect_v": same_grid(grid, reflect_v_grid(grid)),
    }
    return syms


def symmetry_penalty(program, task):
    train_input = task["train"][0]["input"]
    syms = detect_grid_symmetries(train_input)

    penalty = 0.0

    for op_name, params in program:
        if op_name in syms and syms[op_name]:
            penalty += 0.02

    return penalty


def normalize_program(program):
    return [op for op in program if op[0] != "identity"]


def make_hashable(obj):
    if isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    elif isinstance(obj, list):
        return tuple(make_hashable(x) for x in obj)
    else:
        return obj


def program_signature(program):
    norm = normalize_program(program)
    return tuple((op_name, make_hashable(params)) for op_name, params in norm)


def program_complexity(program):
    return len(normalize_program(program))


def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)

    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []

    # initial eval
    for p in beam:
        p = normalize_program(p)
        sig = program_signature(p)

        if sig in seen:
            continue
        seen.add(sig)

        raw_score, pair_scores = eval_program_on_task(task, p)
        adjusted_score = raw_score - symmetry_penalty(p, task)

        ranked.append((adjusted_score, -program_complexity(p), p, pair_scores))

        if raw_score > best_score:
            best_score = raw_score
            best_program = p
            best_pair_scores = pair_scores

    # search loop
    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = normalize_program(program + [op])
                sig = program_signature(new_program)

                if sig in seen:
                    continue
                seen.add(sig)

                raw_score, pair_scores = eval_program_on_task(task, new_program)

                penalty = symmetry_penalty(new_program, task)
                adjusted_score = raw_score - penalty

                candidates.append(
                    (adjusted_score, -program_complexity(new_program), new_program, pair_scores)
                )

                if raw_score > best_score:
                    best_score = raw_score
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        candidate = item[2]
        if candidate != attempt_1:
            attempt_2 = candidate
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }


print("Symmetry-aware beam search loaded.")

Symmetry-aware beam search loaded.


In [ ]:
# ============================================
# 14. RUN PROGRAM (FULL OP SUPPORT, STRICT)
# ============================================

def rotate90_grid(grid):
    return [list(row) for row in zip(*grid[::-1])]

def rotate180_grid(grid):
    return [row[::-1] for row in grid[::-1]]

def rotate270_grid(grid):
    return [list(row) for row in zip(*grid)][::-1]

def reflect_h_grid(grid):
    return grid[::-1]

def reflect_v_grid(grid):
    return [row[::-1] for row in grid]


def tile_n_grid(grid, n):
    out = []
    for row_repeat in range(n):
        for row in grid:
            new_row = []
            for col_repeat in range(n):
                new_row.extend(row)
            out.append(new_row)
    return out


def row_alt_tile_n_grid(grid, n):
    out = []
    for row_repeat in range(n):
        for row in grid:
            new_row = []
            for col_repeat in range(n):
                if col_repeat % 2 == 0:
                    new_row.extend(row)
                else:
                    new_row.extend(row[::-1])
            out.append(new_row)
    return out


def run_program(grid, program):
    current = [row[:] for row in grid]

    for op_name, params in program:

        # ----------------------------------------
        # BASIC OPS
        # ----------------------------------------
        if op_name == "identity":
            current = [row[:] for row in current]

        elif op_name == "color_map":
            mapping = params.get("mapping", {})
            current = [
                [mapping.get(v, v) for v in row]
                for row in current
            ]

        elif op_name == "keep_color":
            color = params.get("color")
            current = [
                [v if v == color else 0 for v in row]
                for row in current
            ]

        # ----------------------------------------
        # GEOMETRIC OPS
        # ----------------------------------------
        elif op_name == "rotate90":
            current = rotate90_grid(current)

        elif op_name == "rotate180":
            current = rotate180_grid(current)

        elif op_name == "rotate270":
            current = rotate270_grid(current)

        elif op_name == "reflect_h":
            current = reflect_h_grid(current)

        elif op_name == "reflect_v":
            current = reflect_v_grid(current)

        elif op_name == "tile_n":
            n = params.get("n", 2)
            current = tile_n_grid(current, n)

        elif op_name == "row_alt_tile_n":
            n = params.get("n", 2)
            current = row_alt_tile_n_grid(current, n)

        # ----------------------------------------
        # SINGLE-OBJECT OPS
        # ----------------------------------------
        elif op_name == "select_role":
            current = op_select_role_v2(
                current,
                role=params.get("role"),
            )

        elif op_name == "recolor_role":
            current = op_recolor_role_v2(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "translate_role":
            current = op_translate_role_v2(
                current,
                role=params.get("role"),
                dx=params.get("dx", 0),
                dy=params.get("dy", 0),
            )

        # ----------------------------------------
        # GROUP OPS
        # ----------------------------------------
        elif op_name == "recolor_group":
            current = op_recolor_group(
                current,
                role=params.get("role"),
                new_color=params.get("color"),
            )

        elif op_name == "select_group":
            current = op_select_group(
                current,
                role=params.get("role"),
            )

        # ----------------------------------------
        # STRICT FAILURE
        # ----------------------------------------
        else:
            raise ValueError(f"Unknown operation in run_program(): {op_name}")

    return current


print("run_program fully restored with geometric + object + group ops.")

run_program fully restored with geometric + object + group ops.


In [ ]:
result = beam_search(task, width=10, depth=3)
debug_full_state(task, result)


🧠 FULL SYSTEM DEBUG OUTPUT

📥 INPUT GRID:
[1, 0]
[0, 1]

📤 TARGET OUTPUT:
[2, 0]
[0, 2]

🔍 OBJECT EXTRACTION:

Detected Objects: 2
------------------------------------------------------------
Object ID: 0
  Color: 1
  Area: 1
  BBox: (0, 0, 0, 0)
  Centroid: (0.0, 0.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(0, 0)]
  Mask:
    [1]
------------------------------------------------------------
Object ID: 1
  Color: 1
  Area: 1
  BBox: (1, 1, 1, 1)
  Centroid: (1.0, 1.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(1, 1)]
  Mask:
    [1]
------------------------------------------------------------

🎭 ROLE ASSIGNMENT:

Inferred Roles
------------------------------------------------------------
largest: Object 0
smallest: Object 1
topmost: Object 0
bottommost: Object 1
leftmost: Object 0
rightmost: Object 1
center_most: Object 0
border_objects: [0, 1]

🧠 BEST PROGRAM:
[('color_map', {'mapping': {0: 0, 1: 2}})]

🎯 ATTEMPT 1:
[('color_map', {'mapping': {0: 0, 

In [ ]:
# ============================================
# 16. ROLE NORMALIZATION + STRICT OBJECT OPS
# ============================================

def normalize_role_name(role):
    role_aliases = {
        "center": "center_most",
        "centre": "center_most",
        "largest_object": "largest",
        "smallest_object": "smallest",
    }
    return role_aliases.get(role, role)


def get_roles_v2(grid, background_color=0, connectivity=4):
    objects = extract_objects(grid, background=background_color, connectivity=connectivity)
    roles = infer_object_roles_v2(objects, grid)
    roles["objects"] = objects
    return roles


def render_object_mask(obj, background_color=0):
    return [row[:] for row in obj["mask"]]


def op_select_role_v2(grid, role, background_color=0, connectivity=4):
    role = normalize_role_name(role)
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [[background_color]]

    obj = roles[role]
    if obj is None:
        return [[background_color]]

    return render_object_mask(obj, background_color=background_color)


def op_recolor_role_v2(grid, role, new_color, background_color=0, connectivity=4):
    role = normalize_role_name(role)
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [row[:] for row in grid]

    obj = roles[role]
    if obj is None:
        return [row[:] for row in grid]

    out = [row[:] for row in grid]

    for r, c in obj["pixels"]:
        out[r][c] = new_color

    return out


def op_translate_role_v2(grid, role, dx, dy, background_color=0, connectivity=4):
    role = normalize_role_name(role)
    roles = get_roles_v2(grid, background_color=background_color, connectivity=connectivity)

    if role not in roles:
        return [row[:] for row in grid]

    obj = roles[role]
    if obj is None:
        return [row[:] for row in grid]

    h, w = len(grid), len(grid[0])
    out = [row[:] for row in grid]

    # erase original object
    for r, c in obj["pixels"]:
        out[r][c] = background_color

    # draw translated object
    for r, c in obj["pixels"]:
        nr = r + dy
        nc = c + dx

        if 0 <= nr < h and 0 <= nc < w:
            out[nr][nc] = obj["color"]

    return out


def safe_get_role(roles, key):
    key = normalize_role_name(key)
    return roles.get(key, None)


def infer_object_recolor_rule(inp, out, background_color=0):
    roles = infer_roles(inp, background_color=background_color)

    candidates = [
        ("largest", safe_get_role(roles, "largest")),
        ("smallest", safe_get_role(roles, "smallest")),
        ("center_most", safe_get_role(roles, "center_most")),
    ]

    for role_name, obj in candidates:
        if obj is None:
            continue

        changed_colors = set()
        changed_elsewhere = False
        pixel_set = set(obj["pixels"])

        for r in range(len(inp)):
            for c in range(len(inp[0])):
                if inp[r][c] != out[r][c]:
                    if (r, c) in pixel_set:
                        changed_colors.add(out[r][c])
                    else:
                        changed_elsewhere = True

        if not changed_elsewhere and len(changed_colors) == 1:
            return {"role": role_name, "color": list(changed_colors)[0]}

    return None


print("Role normalization + strict object ops loaded.")

Role normalization + strict object ops loaded.


In [ ]:
# ============================================
# 18. TASK ARCHETYPE DETECTOR
# ============================================

def detect_task_archetype(task):
    pair = task["train"][0]
    inp = pair["input"]
    out = pair["output"]

    # ----------------------------------------
    # SHAPE CHANGE
    # ----------------------------------------
    if len(inp) != len(out) or len(inp[0]) != len(out[0]):
        return "scaling_or_tiling"

    # ----------------------------------------
    # COLOR ANALYSIS
    # ----------------------------------------
    inp_colors = set(v for row in inp for v in row)
    out_colors = set(v for row in out for v in row)

    # detect mapping
    mapping = {}
    valid_map = True

    for r in range(len(inp)):
        for c in range(len(inp[0])):
            a = inp[r][c]
            b = out[r][c]

            if a in mapping and mapping[a] != b:
                valid_map = False
                break
            mapping[a] = b

    if valid_map and len(mapping) > 1:
        return "color_map"

    # ----------------------------------------
    # OBJECT-BASED CHANGE
    # ----------------------------------------
    objs_in = extract_objects(inp)
    objs_out = extract_objects(out)

    if len(objs_in) == len(objs_out):
        # check movement
        moved = False
        for o1, o2 in zip(objs_in, objs_out):
            if o1["centroid"] != o2["centroid"]:
                moved = True

        if moved:
            return "object_motion"

    # ----------------------------------------
    # DEFAULT
    # ----------------------------------------
    return "mixed"


print("Task archetype detector ready.")

Task archetype detector ready.


In [ ]:
# ============================================
# 19. ARCHETYPE-AWARE SCORING
# ============================================

def archetype_penalty(program, task):
    archetype = detect_task_archetype(task)

    penalty = 0.0

    for op_name, _ in program:

        if archetype == "color_map":
            if op_name not in ["color_map", "identity"]:
                penalty += 0.05

        elif archetype == "object_motion":
            if op_name not in ["translate_role", "select_role"]:
                penalty += 0.05

        elif archetype == "scaling_or_tiling":
            if op_name not in ["tile_n", "row_alt_tile_n"]:
                penalty += 0.05

    return penalty


print("Archetype scoring ready.")

Archetype scoring ready.


In [ ]:
# ============================================
# 20. FINAL BEAM SEARCH (ARCHETYPE + SYMMETRY)
# ============================================

def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)

    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []

    for p in beam:
        p = normalize_program(p)
        sig = program_signature(p)

        if sig in seen:
            continue
        seen.add(sig)

        raw_score, pair_scores = eval_program_on_task(task, p)

        penalty = symmetry_penalty(p, task) + archetype_penalty(p, task)
        score = raw_score - penalty

        ranked.append((score, -program_complexity(p), p, pair_scores))

        if raw_score > best_score:
            best_score = raw_score
            best_program = p
            best_pair_scores = pair_scores

    for _ in range(depth):
        candidates = []

        for _, _, program, _ in ranked[:width]:
            for op in ops:
                new_program = normalize_program(program + [op])
                sig = program_signature(new_program)

                if sig in seen:
                    continue
                seen.add(sig)

                raw_score, pair_scores = eval_program_on_task(task, new_program)

                penalty = symmetry_penalty(new_program, task) + archetype_penalty(new_program, task)
                score = raw_score - penalty

                candidates.append(
                    (score, -program_complexity(new_program), new_program, pair_scores)
                )

                if raw_score > best_score:
                    best_score = raw_score
                    best_program = new_program
                    best_pair_scores = pair_scores

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        if item[2] != attempt_1:
            attempt_2 = item[2]
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
    }


print("Final beam search with archetype + symmetry loaded.")

Final beam search with archetype + symmetry loaded.


In [ ]:
result = beam_search(task, width=10, depth=3)
debug_full_state(task, result)


🧠 FULL SYSTEM DEBUG OUTPUT

📥 INPUT GRID:
[1, 0]
[0, 1]

📤 TARGET OUTPUT:
[2, 0]
[0, 2]

🔍 OBJECT EXTRACTION:

Detected Objects: 2
------------------------------------------------------------
Object ID: 0
  Color: 1
  Area: 1
  BBox: (0, 0, 0, 0)
  Centroid: (0.0, 0.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(0, 0)]
  Mask:
    [1]
------------------------------------------------------------
Object ID: 1
  Color: 1
  Area: 1
  BBox: (1, 1, 1, 1)
  Centroid: (1.0, 1.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(1, 1)]
  Mask:
    [1]
------------------------------------------------------------

🎭 ROLE ASSIGNMENT:

Inferred Roles
------------------------------------------------------------
largest: Object 0
smallest: Object 1
topmost: Object 0
bottommost: Object 1
leftmost: Object 0
rightmost: Object 1
center_most: Object 0
border_objects: [0, 1]

🧠 BEST PROGRAM:
[('color_map', {'mapping': {0: 0, 1: 2}})]

🎯 ATTEMPT 1:
[('color_map', {'mapping': {0: 0, 

In [ ]:
debug_full_state(task, result)


🧠 FULL SYSTEM DEBUG OUTPUT

📥 INPUT GRID:
[1, 0]
[0, 1]

📤 TARGET OUTPUT:
[2, 0]
[0, 2]

🔍 OBJECT EXTRACTION:

Detected Objects: 2
------------------------------------------------------------
Object ID: 0
  Color: 1
  Area: 1
  BBox: (0, 0, 0, 0)
  Centroid: (0.0, 0.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(0, 0)]
  Mask:
    [1]
------------------------------------------------------------
Object ID: 1
  Color: 1
  Area: 1
  BBox: (1, 1, 1, 1)
  Centroid: (1.0, 1.0)
  Width x Height: 1 x 1
  Touches Border: True
  Pixels: [(1, 1)]
  Mask:
    [1]
------------------------------------------------------------

🎭 ROLE ASSIGNMENT:

Inferred Roles
------------------------------------------------------------
largest: Object 0
smallest: Object 1
topmost: Object 0
bottommost: Object 1
leftmost: Object 0
rightmost: Object 1
center_most: Object 0
border_objects: [0, 1]

🧠 BEST PROGRAM:
[('color_map', {'mapping': {0: 0, 1: 2}})]

🎯 ATTEMPT 1:
[('color_map', {'mapping': {0: 0, 

In [115]:
# ============================================
# AOFE v13.5 MINIMAL INTEGRATION PATCH
# Adds:
# - analyze_errors
# - calculate_aofe_score
# - eval_program_with_errors
# - get_priority_ops
# - beam_search override with density-guided refinement
# ============================================

def analyze_errors(pred, target):
    same_shape = (len(pred) == len(target) and len(pred[0]) == len(target[0]))

    h = max(len(pred), len(target))
    w = max(len(pred[0]), len(target[0]))

    error_map = []
    error_count = 0

    for r in range(h):
        row = []
        for c in range(w):
            pv = pred[r][c] if r < len(pred) and c < len(pred[0]) else None
            tv = target[r][c] if r < len(target) and c < len(target[0]) else None
            mismatch = int(pv != tv)
            row.append(mismatch)
            error_count += mismatch
        error_map.append(row)

    total_cells = len(target) * len(target[0]) if same_shape else h * w
    error_density = error_count / max(1, total_cells)

    if same_shape:
        pixel_accuracy = 1.0 - (error_count / max(1, len(target) * len(target[0])))
    else:
        pixel_accuracy = 0.0

    pred_colors = set(v for row in pred for v in row)
    tgt_colors = set(v for row in target for v in row)
    color_score = len(pred_colors & tgt_colors) / max(1, len(pred_colors | tgt_colors))

    try:
        pred_objs = extract_objects(pred)
        tgt_objs = extract_objects(target)
        structural_count_score = 1.0 - (
            abs(len(pred_objs) - len(tgt_objs)) / max(1, max(len(pred_objs), len(tgt_objs)))
        )
    except Exception:
        structural_count_score = 0.0

    structural_score = 0.5 * float(same_shape) + 0.5 * structural_count_score

    return {
        "same_shape": same_shape,
        "error_map": error_map,
        "error_count": error_count,
        "error_density": max(0.0, min(1.0, error_density)),
        "pixel_accuracy": max(0.0, min(1.0, pixel_accuracy)),
        "color_score": max(0.0, min(1.0, color_score)),
        "structural_score": max(0.0, min(1.0, structural_score)),
    }


def calculate_aofe_score(pred, target):
    a = analyze_errors(pred, target)

    # AOFE v13.5 weighting:
    # pixel 60%, structure 25%, color 15%
    score = (
        0.60 * a["pixel_accuracy"] +
        0.25 * a["structural_score"] +
        0.15 * a["color_score"]
    )

    return max(0.0, min(1.0, score))


def eval_program_with_errors(task, program):
    analyses = []
    pair_scores = []

    for pair in task["train"]:
        pred = run_program(pair["input"], program)
        analysis = analyze_errors(pred, pair["output"])
        score = calculate_aofe_score(pred, pair["output"])

        analyses.append(analysis)
        pair_scores.append(score)

    avg_score = sum(pair_scores) / max(1, len(pair_scores))
    return avg_score, analyses


# ============================================
# AOFE PRIORITY OPS (DIVERSITY-AWARE)
# ============================================

def get_priority_ops(error_analysis, ops):
    error_density = error_analysis.get("error_density", 0.0)

    # categorize ops
    transform_ops = []
    color_ops = []
    object_ops = []
    other_ops = []

    for op in ops:
        name = op[0]

        if "rotate" in name or "flip" in name:
            transform_ops.append(op)
        elif "translate" in name:
            transform_ops.append(op)
        elif "color" in name:
            color_ops.append(op)
        elif "recolor" in name or "role" in name:
            object_ops.append(op)
        else:
            other_ops.append(op)

    # ----------------------------------------
    # DIVERSITY STRATEGY
    # ----------------------------------------

    # 🚨 HIGH ERROR → force exploration
    if error_density > 0.2:
        ordered = (
            transform_ops[:3] +
            object_ops[:3] +
            color_ops[:2] +
            other_ops
        )

    # 🚨 MID ERROR → mix strategies
    elif error_density > 0.05:
        ordered = (
            object_ops[:3] +
            transform_ops[:3] +
            color_ops[:2] +
            other_ops
        )

    # 🚨 LOW ERROR → fine tuning
    else:
        ordered = (
            object_ops[:2] +
            color_ops[:2] +
            transform_ops[:2] +
            other_ops
        )

    return ordered

def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)
    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    def program_len(p):
        return len(normalize_program(p))

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores, best_analyses, best_refinement_steps

        p = normalize_program(program)
        sig = program_signature(p)

        if sig in seen:
            return None
        seen.add(sig)

        raw_score, analyses = eval_program_with_errors(task, p)

        penalty = symmetry_penalty(p, task) + archetype_penalty(p, task)
        final_score = raw_score - penalty

        pair_scores = []
        for pair in task["train"]:
            pred = run_program(pair["input"], p)
            pair_scores.append(calculate_aofe_score(pred, pair["output"]))

        item = (
            final_score,
            -program_complexity(p),   # MDL tie-break
            p,
            pair_scores,
            analyses,
            refinement_steps,
        )

        if (
            raw_score > best_score or
            (raw_score == best_score and best_program is not None and program_len(p) < program_len(best_program)) or
            (raw_score == best_score and best_program is None)
        ):
            best_program = p
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return item

    # ----------------------------------------
    # Seed stage
    # ----------------------------------------
    for p in beam:
        item = consider(p, refinement_steps=0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    # ----------------------------------------
    # Refinement loop
    # ----------------------------------------
    for step in range(depth):
        candidates = []

        for _, _, program, _, analyses, _ in ranked[:width]:
            if not analyses:
                continue

            worst_analysis = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst_analysis, ops)

            # density-priority expansion
            for op in ordered_ops[:width]:
                new_program = normalize_program(program + [op])
                item = consider(new_program, refinement_steps=step + 1)
                if item is not None:
                    candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        cand = item[2]
        if cand != attempt_1:
            attempt_2 = cand
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [
            round(a["error_density"], 4) for a in best_analyses
        ],
        "failure_mode": None if best_score == 1.0 else "near_miss_or_missing_operator",
    }


print("AOFE v13.5 minimal integration patch loaded.")

AOFE v13.5 minimal integration patch loaded.


In [116]:
# ============================================
# AOFE v13.5 VERIFICATION CELL
# Confirms wiring + runs 3 controlled tests
# ============================================

import json
import traceback
from copy import deepcopy

# --------------------------------------------
# REQUIRED FUNCTION CHECK
# --------------------------------------------
REQUIRED_FUNCS = [
    "run_program",
    "extract_objects",
    "infer_object_roles_v2",
    "analyze_errors",
    "eval_program_with_errors",
    "calculate_aofe_score",
    "get_priority_ops",
    "beam_search",
]

print("=" * 70)
print("AOFE v13.5 FUNCTION CONNECTIVITY CHECK")
print("=" * 70)

missing = []
for fn_name in REQUIRED_FUNCS:
    if fn_name in globals() and callable(globals()[fn_name]):
        print(f"[OK] {fn_name}")
    else:
        print(f"[MISSING] {fn_name}")
        missing.append(fn_name)

if missing:
    raise RuntimeError(f"Missing required functions: {missing}")

print("\nAll required functions are active.\n")

# --------------------------------------------
# HELPER: PRETTY PROGRAM
# --------------------------------------------
def format_program(program):
    try:
        return " -> ".join(
            f"{op}{'' if not params else '(' + ', '.join(f'{k}={v}' for k, v in params.items()) + ')'}"
            for op, params in program
        ) if program else "[]"
    except Exception:
        return str(program)

# --------------------------------------------
# HELPER: SAFE TASK RUN
# --------------------------------------------
def run_controlled_task(task_name, task, width=10, depth=3):
    print("=" * 70)
    print(f"TEST: {task_name}")
    print("=" * 70)

    try:
        result = beam_search(task, width=width, depth=depth)

        best_program = result.get("best_program")
        best_score = result.get("best_score")
        refinement_steps = result.get("refinement_steps")
        density_prog = result.get("error_density_progression")
        failure_mode = result.get("failure_mode")

        print("Best program:")
        print(format_program(best_program))
        print(f"Score: {best_score:.4f}")
        print(f"Refinement steps: {refinement_steps}")
        print(f"Error density progression: {density_prog}")
        print(f"Failure mode: {failure_mode}")

        # Per-train-pair detailed check
        print("\nPer-train-pair evaluation:")
        for i, pair in enumerate(task["train"]):
            pred = run_program(pair["input"], best_program)
            analysis = analyze_errors(pred, pair["output"])
            score = calculate_aofe_score(pred, pair["output"])

            print(f"  Pair {i+1}:")
            print(f"    score = {score:.4f}")
            print(f"    pixel_accuracy = {analysis['pixel_accuracy']:.4f}")
            print(f"    structural_score = {analysis['structural_score']:.4f}")
            print(f"    color_score = {analysis['color_score']:.4f}")
            print(f"    error_density = {analysis['error_density']:.4f}")
            print(f"    error_count = {analysis['error_count']}")

        print()

        return {
            "task_name": task_name,
            "status": "ok",
            "best_program": best_program,
            "best_score": best_score,
            "refinement_steps": refinement_steps,
            "error_density_progression": density_prog,
            "failure_mode": failure_mode,
        }

    except Exception as e:
        print("ERROR during test:")
        traceback.print_exc()
        print()
        return {
            "task_name": task_name,
            "status": "error",
            "error": str(e),
        }

# --------------------------------------------
# CONTROLLED TASK 1: SIMPLE RECOLOR
# input -> global recolor 1->2
# --------------------------------------------
task_recolor = {
    "train": [
        {
            "input": [
                [0,0,0],
                [0,1,0],
                [0,0,0],
            ],
            "output": [
                [0,0,0],
                [0,2,0],
                [0,0,0],
            ],
        },
        {
            "input": [
                [1,0,1],
                [0,0,0],
                [1,0,1],
            ],
            "output": [
                [2,0,2],
                [0,0,0],
                [2,0,2],
            ],
        },
    ],
    "test": [
        {
            "input": [
                [0,1,0],
                [1,0,1],
                [0,1,0],
            ]
        }
    ]
}

# --------------------------------------------
# CONTROLLED TASK 2: ROTATION / TRANSFORMATION
# L-shape rotated 90 degrees
# --------------------------------------------
task_rotation = {
    "train": [
        {
            "input": [
                [3,0,0],
                [3,0,0],
                [3,3,0],
            ],
            "output": [
                [3,3,3],
                [3,0,0],
                [0,0,0],
            ],
        },
        {
            "input": [
                [4,0,0],
                [4,4,4],
                [0,0,0],
            ],
            "output": [
                [0,4,4],
                [0,4,0],
                [0,4,0],
            ],
        },
    ],
    "test": [
        {
            "input": [
                [5,0,0],
                [5,0,0],
                [5,5,0],
            ]
        }
    ]
}

# --------------------------------------------
# CONTROLLED TASK 3: OBJECT-BASED TASK
# largest object gets recolored to 8
# smaller object stays unchanged
# --------------------------------------------
task_object = {
    "train": [
        {
            "input": [
                [1,1,0,2],
                [1,1,0,0],
                [0,0,0,0],
                [0,0,0,0],
            ],
            "output": [
                [8,8,0,2],
                [8,8,0,0],
                [0,0,0,0],
                [0,0,0,0],
            ],
        },
        {
            "input": [
                [0,3,0,0],
                [0,3,0,4],
                [0,3,0,4],
                [0,3,0,0],
            ],
            "output": [
                [0,8,0,0],
                [0,8,0,4],
                [0,8,0,4],
                [0,8,0,0],
            ],
        },
    ],
    "test": [
        {
            "input": [
                [5,5,5,0],
                [0,0,0,6],
                [0,0,0,6],
                [0,0,0,0],
            ]
        }
    ]
}

# --------------------------------------------
# RUN TESTS
# --------------------------------------------
summary = []

summary.append(run_controlled_task("simple recolor", task_recolor, width=10, depth=3))
summary.append(run_controlled_task("rotation / transformation", task_rotation, width=12, depth=4))
summary.append(run_controlled_task("object-based task", task_object, width=12, depth=4))

# --------------------------------------------
# FINAL SUMMARY
# --------------------------------------------
print("=" * 70)
print("AOFE v13.5 CONTROLLED TEST SUMMARY")
print("=" * 70)

for item in summary:
    if item["status"] == "ok":
        print(f"- {item['task_name']}")
        print(f"  status: OK")
        print(f"  best_score: {item['best_score']:.4f}")
        print(f"  refinement_steps: {item['refinement_steps']}")
        print(f"  error_density_progression: {item['error_density_progression']}")
        print(f"  failure_mode: {item['failure_mode']}")
        print(f"  best_program: {format_program(item['best_program'])}")
    else:
        print(f"- {item['task_name']}")
        print(f"  status: ERROR")
        print(f"  error: {item['error']}")

    print()

print("Verification run complete.")

AOFE v13.5 FUNCTION CONNECTIVITY CHECK
[OK] run_program
[OK] extract_objects
[OK] infer_object_roles_v2
[OK] analyze_errors
[OK] eval_program_with_errors
[OK] calculate_aofe_score
[OK] get_priority_ops
[OK] beam_search

All required functions are active.

TEST: simple recolor
Best program:
color_map(mapping={0: 0, 1: 2})
Score: 1.0000
Refinement steps: 0
Error density progression: [0.0, 0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0
  Pair 2:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0

TEST: rotation / transformation
Best program:
rotate90
Score: 1.0000
Refinement steps: 1
Error density progression: [0.0, 0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    stru

In [117]:
# ============================================
# HARD TASK 1: PARTIAL MATCH (FORCES REFINEMENT)
# ============================================

task_partial = {
    "train": [
        {
            "input": [
                [1,1,0],
                [1,0,0],
                [0,0,0],
            ],
            "output": [
                [2,2,0],
                [2,0,0],
                [0,0,0],
            ],
        },
        {
            "input": [
                [1,1,1],
                [0,1,0],
                [0,0,0],
            ],
            "output": [
                [2,2,2],
                [0,2,0],
                [0,0,0],
            ],
        },
    ],
    "test": [{"input": [[1,0,1],[0,1,0],[1,0,1]]}]
}


# ============================================
# HARD TASK 2: WRONG FIRST OPERATOR
# ============================================

task_misleading = {
    "train": [
        {
            "input": [
                [3,0,0],
                [3,3,0],
                [0,0,0],
            ],
            "output": [
                [0,0,3],
                [0,3,3],
                [0,0,0],
            ],
        }
    ],
    "test": [{"input": [[4,0,0],[4,4,0],[0,0,0]]}]
}


# ============================================
# HARD TASK 3: OBJECT VS GLOBAL CONFLICT
# ============================================

task_conflict = {
    "train": [
        {
            "input": [
                [1,1,0],
                [0,2,2],
                [0,0,0],
            ],
            "output": [
                [3,3,0],
                [0,2,2],
                [0,0,0],
            ],
        }
    ],
    "test": [{"input": [[4,4,0],[0,5,5],[0,0,0]]}]
}


# ============================================
# RUN HARD TESTS
# ============================================

print("\n" + "="*70)
print("AOFE HARD TEST SUITE")
print("="*70)

run_controlled_task("partial recolor (forces refinement)", task_partial, width=12, depth=4)
run_controlled_task("misleading transformation", task_misleading, width=12, depth=4)
run_controlled_task("object vs global conflict", task_conflict, width=12, depth=4)


AOFE HARD TEST SUITE
TEST: partial recolor (forces refinement)
Best program:
color_map(mapping={1: 2, 0: 0})
Score: 1.0000
Refinement steps: 0
Error density progression: [0.0, 0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0
  Pair 2:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0

TEST: misleading transformation
Best program:
reflect_v
Score: 1.0000
Refinement steps: 1
Error density progression: [0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0

TEST: object vs global conflict
Best program:
color_map(mapping={1: 3, 0: 0, 2: 2})
Score: 1.0000
Refinement steps: 0
Erro

{'task_name': 'object vs global conflict',
 'status': 'ok',
 'best_program': [('color_map', {'mapping': {1: 3, 0: 0, 2: 2}})],
 'best_score': 1.0,
 'refinement_steps': 0,
 'error_density_progression': [0.0],
 'failure_mode': None}

In [118]:
# ============================================
# AOFE v13.5 BEAM SEARCH (CLEAN + FIXED)
# ============================================

def beam_search(task, width=10, depth=3):
    ops = build_task_ops(task)
    beam = seed_programs_from_reasoning(task)
    seen = set()

    ranked = []
    best_program = None
    best_score = -1.0
    best_pair_scores = []
    best_analyses = []
    best_refinement_steps = 0

    def program_len(p):
        return len(normalize_program(p))

    def consider(program, refinement_steps):
        nonlocal best_program, best_score, best_pair_scores, best_analyses, best_refinement_steps

        p = normalize_program(program)
        sig = program_signature(p)

        if sig in seen:
            return None
        seen.add(sig)

        raw_score, analyses = eval_program_with_errors(task, p)

        penalty = symmetry_penalty(p, task) + archetype_penalty(p, task)
        final_score = raw_score - penalty

        pair_scores = []
        for pair in task["train"]:
            pred = run_program(pair["input"], p)
            pair_scores.append(calculate_aofe_score(pred, pair["output"]))

        item = (
            final_score,
            -program_complexity(p),   # MDL tie-break
            p,
            pair_scores,
            analyses,
            refinement_steps,
        )

        # BEST PROGRAM TRACKING (MDL-aware)
        if (
            raw_score > best_score or
            (raw_score == best_score and best_program is not None and program_len(p) < program_len(best_program)) or
            (raw_score == best_score and best_program is None)
        ):
            best_program = p
            best_score = raw_score
            best_pair_scores = pair_scores
            best_analyses = analyses
            best_refinement_steps = refinement_steps

        return item

    # ----------------------------------------
    # SEED STAGE
    # ----------------------------------------
    for p in beam:
        item = consider(p, refinement_steps=0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    # ----------------------------------------
    # REFINEMENT LOOP (FIXED + CLEAN)
    # ----------------------------------------
    for step in range(depth):
        candidates = []

        for _, _, program, _, analyses, _ in ranked[:width]:
            if not analyses:
                continue

            worst_analysis = max(analyses, key=lambda a: a["error_density"])
            ordered_ops = get_priority_ops(worst_analysis, ops)

            last_op = program[-1][0] if program else None

            # 🚨 PRIMARY: avoid repeating same operator
            for op in ordered_ops[:width]:

                if op[0] == last_op:
                    continue

                new_program = normalize_program(program + [op])
                item = consider(new_program, refinement_steps=step + 1)

                if item is not None:
                    candidates.append(item)

            # 🚨 SECONDARY: allow LIMITED repetition (fallback)
            for op in ordered_ops[:2]:
                if op[0] == last_op:
                    new_program = normalize_program(program + [op])
                    item = consider(new_program, refinement_steps=step + 1)
                    if item is not None:
                        candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score == 1.0:
            break

    # ----------------------------------------
    # FINAL SELECTION
    # ----------------------------------------
    attempt_1 = best_program
    attempt_2 = best_program

    for item in ranked:
        cand = item[2]
        if cand != attempt_1:
            attempt_2 = cand
            break

    return {
        "best_program": best_program,
        "best_score": best_score,
        "pair_scores": best_pair_scores,
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
        "ranked": ranked[:width],
        "refinement_steps": best_refinement_steps,
        "error_density_progression": [
            round(a["error_density"], 4) for a in best_analyses
        ],
        "failure_mode": None if best_score == 1.0 else "near_miss_or_missing_operator",
    }


print("AOFE beam_search (fixed + anti-loop) loaded.")

AOFE beam_search (fixed + anti-loop) loaded.


In [119]:
run_controlled_task("misleading transformation", task_misleading, width=12, depth=4)

TEST: misleading transformation
Best program:
reflect_v
Score: 1.0000
Refinement steps: 1
Error density progression: [0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0



{'task_name': 'misleading transformation',
 'status': 'ok',
 'best_program': [('reflect_v', {})],
 'best_score': 1.0,
 'refinement_steps': 1,
 'error_density_progression': [0.0],
 'failure_mode': None}

In [120]:
# ============================================
# HARD MULTI-STEP TASK (CHAINED REASONING)
# ============================================

task_chain = {
    "train": [
        {
            "input": [
                [1,0,0],
                [1,1,0],
                [0,0,0],
            ],
            "output": [
                [0,0,2],
                [0,2,2],
                [0,0,0],
            ],
        }
    ],
    "test": [{"input": [[3,0,0],[3,3,0],[0,0,0]]}]
}

run_controlled_task("multi-step chained transform", task_chain, width=15, depth=5)

TEST: multi-step chained transform
Best program:
recolor_role(role=largest, color=2) -> reflect_v
Score: 1.0000
Refinement steps: 2
Error density progression: [0.0]
Failure mode: None

Per-train-pair evaluation:
  Pair 1:
    score = 1.0000
    pixel_accuracy = 1.0000
    structural_score = 1.0000
    color_score = 1.0000
    error_density = 0.0000
    error_count = 0



{'task_name': 'multi-step chained transform',
 'status': 'ok',
 'best_program': [('recolor_role', {'role': 'largest', 'color': 2}),
  ('reflect_v', {})],
 'best_score': 1.0,
 'refinement_steps': 2,
 'error_density_progression': [0.0],
 'failure_mode': None}

In [123]:
# ============================================
# LOAD REAL ARC DATASET
# ============================================

import json

DATA_PATH = "/content/data/raw/arc_tasks.json"

tasks = json.load(open(DATA_PATH))

print("Loaded tasks:", len(tasks))
print("Sample keys:", list(tasks.keys())[:5])

Loaded tasks: 1
Sample keys: ['demo_task']


In [124]:
# ============================================
# FILE DEDUP + SAFE RENAMING
# ============================================

import os
import hashlib
import shutil

SEARCH_DIRS = [
    "/content",
    "/content/data",
    "/content/data/raw"
]

def file_hash(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

# collect files by name
files_by_name = {}

for base in SEARCH_DIRS:
    if not os.path.exists(base):
        continue

    for root, _, files in os.walk(base):
        for f in files:
            full = os.path.join(root, f)
            files_by_name.setdefault(f, []).append(full)

print("=" * 60)
print("DUPLICATE FILE ANALYSIS")
print("=" * 60)

renamed = []

for name, paths in files_by_name.items():
    if len(paths) <= 1:
        continue

    print(f"\n🔁 Duplicate filename detected: {name}")

    hashes = {}

    for p in paths:
        h = file_hash(p)
        hashes.setdefault(h, []).append(p)

    # if all hashes same → true duplicate
    if len(hashes) == 1:
        print("  ✔ Identical files (safe, no rename needed)")
        continue

    # otherwise rename each version
    print("  ⚠ Different content detected → renaming")

    for i, (h, group) in enumerate(hashes.items()):
        for j, path in enumerate(group):
            base, ext = os.path.splitext(path)

            new_name = f"{base}__v{i+1}_{j+1}{ext}"

            shutil.move(path, new_name)
            renamed.append((path, new_name))

            print(f"  RENAMED:\n    {path}\n    → {new_name}")

print("\n" + "=" * 60)
print("RENAMING COMPLETE")
print("=" * 60)

print(f"\nTotal files renamed: {len(renamed)}")

DUPLICATE FILE ANALYSIS

🔁 Duplicate filename detected: README.md
  ⚠ Different content detected → renaming
  RENAMED:
    /content/README.md
    → /content/README__v1_1.md
  RENAMED:
    /content/sample_data/README.md
    → /content/sample_data/README__v2_1.md

🔁 Duplicate filename detected: arc_tasks.json
  ✔ Identical files (safe, no rename needed)

RENAMING COMPLETE

Total files renamed: 2


In [125]:
# ============================================
# FULL FILE CONTENT INSPECTION
# ============================================

import os
import json

SEARCH_DIR = "/content"

def inspect_file(path):
    print("\n" + "="*60)
    print("FILE:", path)

    try:
        if path.endswith(".json"):
            with open(path, "r") as f:
                data = json.load(f)

            print("Type:", type(data))

            if isinstance(data, dict):
                keys = list(data.keys())
                print("Keys:", keys[:5])

                # ARC multi-task format
                if len(keys) > 0 and isinstance(data[keys[0]], dict):
                    sample = data[keys[0]]
                    if "train" in sample and "test" in sample:
                        print("👉 ARC MULTI-TASK DATASET (GOOD)")

                # single ARC task
                if "train" in data and "test" in data:
                    print("👉 SINGLE ARC TASK (DEMO)")

                # memory DB
                if "memory" in keys or "patterns" in keys:
                    print("👉 MEMORY / PATTERN DB")

            elif isinstance(data, list):
                print("List length:", len(data))
                if len(data) > 0 and isinstance(data[0], dict):
                    print("Sample keys:", list(data[0].keys())[:5])

                    if "input" in data[0] and "output" in data[0]:
                        print("👉 POSSIBLE TRAINING PAIRS (but NOT ARC format)")
                    elif "trace" in data[0] or "steps" in data[0]:
                        print("👉 TRACE DATASET")

        elif path.endswith(".txt"):
            with open(path, "r", errors="ignore") as f:
                content = f.read(500)

            print("Text preview:")
            print(content[:300])

    except Exception as e:
        print("ERROR reading file:", e)


# walk all files
for root, _, files in os.walk(SEARCH_DIR):
    for f in files:
        full = os.path.join(root, f)
        inspect_file(full)


FILE: /content/[ARC_AGI_2]_EDA_+_Training_Qwen3_baseline (5).ipynb

FILE: /content/qwen3token (1).txt
Text preview:
{
  "version": "1.0",
  "truncation": null,
  "padding": null,
  "added_tokens": [
    {
      "id": 151643,
      "content": "<|endoftext|>",
      "single_word": false,
      "lstrip": false,
      "rstrip": false,
      "normalized": false,
      "special": true
    },
    {
      "id": 151644,
 

FILE: /content/cells_chunk_3.json
Type: <class 'list'>
List length: 50
Sample keys: ['cell_id', 'code', 'terms', 'methodology', 'output']

FILE: /content/arc_trace_dataset.json
Type: <class 'list'>
List length: 1000
Sample keys: ['task_id', 'signature', 'grammar', 'explanation']

FILE: /content/Qwen3-main.zip

FILE: /content/arc-agi-2-eda-training-qwen3-baseline (2).ipynb

FILE: /content/training_args.bin

FILE: /content/arc_compositional_memory_db.json
Type: <class 'list'>
List length: 158
Sample keys: ['task_id', 'signature', 'rule_template', 'parameters', 'pattern_type']


In [126]:
import json

DATA_PATH = "/content/arc-agi_training_challenges.json"

tasks = json.load(open(DATA_PATH))

print("Loaded tasks:", len(tasks))
print("Sample keys:", list(tasks.keys())[:5])

Loaded tasks: 1000
Sample keys: ['00576224', '007bbfb7', '009d5c81', '00d62c1b', '00dbd492']


In [127]:
solver_tasks = convert_to_solver_format(tasks)
print(len(solver_tasks))

1000


In [129]:
solver_task_list = list(solver_tasks.values())

In [130]:
for i in range(5):
    print("\n" + "="*70)
    print(f"TASK {i}")
    print("="*70)

    run_controlled_task(
        f"ARC Task {i}",
        solver_task_list[i],
        width=15,
        depth=5
    )


TASK 0
TEST: ARC Task 0
Best program:
rotate90 -> row_alt_tile_n(n=3) -> rotate270
Score: 0.8167
Refinement steps: 3
Error density progression: [0.3333, 0.1667]
Failure mode: near_miss_or_missing_operator

Per-train-pair evaluation:
  Pair 1:
    score = 0.7583
    pixel_accuracy = 0.6667
    structural_score = 0.8333
    color_score = 1.0000
    error_density = 0.3333
    error_count = 12
  Pair 2:
    score = 0.8750
    pixel_accuracy = 0.8333
    structural_score = 0.9000
    color_score = 1.0000
    error_density = 0.1667
    error_count = 6


TASK 1
TEST: ARC Task 1
Best program:
rotate270 -> row_alt_tile_n(n=3) -> rotate90
Score: 0.8303
Refinement steps: 3
Error density progression: [0.2963, 0.2222, 0.2716, 0.2469, 0.1728]
Failure mode: near_miss_or_missing_operator

Per-train-pair evaluation:
  Pair 1:
    score = 0.7910
    pixel_accuracy = 0.7037
    structural_score = 0.8750
    color_score = 1.0000
    error_density = 0.2963
    error_count = 24
  Pair 2:
    score = 0.8220

In [131]:
def tile_grid_n(grid, n):
    grid = np.array(grid)
    return np.tile(grid, (n, n))

In [132]:
("tile_grid_n", {"n": 2})
("tile_grid_n", {"n": 3})

('tile_grid_n', {'n': 3})

In [133]:
def conditional_color_map(grid, condition_color, new_color):
    grid = np.array(grid)
    out = grid.copy()
    out[grid == condition_color] = new_color
    return out

In [134]:
("conditional_color_map", {"condition_color": 1, "new_color": 2})

('conditional_color_map', {'condition_color': 1, 'new_color': 2})

In [137]:
def get_priority_ops(analysis, ops):
    """
    Returns prioritized operator list based on error analysis
    """

    priority = []

    # ----------------------------------
    # BASE PRIORITY (existing behavior)
    # ----------------------------------
    if analysis["error_density"] > 0.2:
        priority.extend(["translate_role", "move_object"])

    if analysis["pixel_accuracy"] < 0.9:
        priority.extend(["color_map", "recolor_role"])

    if analysis["structural_score"] < 0.9:
        priority.extend(["rotate90", "reflect_v", "reflect_h"])

    # ----------------------------------
    # 🔥 NEW: STRUCTURE OK BUT PIXELS WRONG
    # ----------------------------------
    if analysis["structural_score"] > 0.8 and analysis["pixel_accuracy"] < 1.0:
        priority.extend(["recolor_role", "conditional_color_map"])

    # ----------------------------------
    # CLEAN + ORDER
    # ----------------------------------
    seen = set()
    ordered = []

    for p in priority:
        for op in ops:
            if op[0] == p and op[0] not in seen:
                ordered.append(op)
                seen.add(op[0])

    # fallback: include remaining ops
    for op in ops:
        if op[0] not in seen:
            ordered.append(op)

    return ordered

In [140]:
# ============================================
# AOFE v13.5 REAL ARC EVALUATION (5 TASKS)
# ============================================

import json

# ----------------------------------
# LOAD DATASET
# ----------------------------------
DATA_PATH = "/content/arc-agi_training_challenges.json"

tasks = json.load(open(DATA_PATH))
print("Loaded tasks:", len(tasks))

# ----------------------------------
# CONVERT TO SOLVER FORMAT
# ----------------------------------
solver_tasks = convert_to_solver_format(tasks)

print("Converted tasks:", len(solver_tasks))


# ----------------------------------
# RUN FIRST 5 TASKS
# ----------------------------------
results = []

for i, (task_id, task) in enumerate(solver_tasks.items()):
    if i >= 5:
        break

    print("\n" + "="*70)
    print(f"TASK {i} | ID: {task_id}")
    print("="*70)

    result = run_controlled_task(
        f"ARC Task {task_id}",
        task,
        width=15,
        depth=5
    )

    results.append({
        "task_id": task_id,
        "result": result
    })


# ----------------------------------
# SUMMARY
# ----------------------------------
print("\n" + "="*70)
print("AOFE v13.5 SUMMARY (FIRST 5 TASKS)")
print("="*70)

for r in results:
    task_id = r["task_id"]
    res = r["result"]

    print(f"\nTask: {task_id}")
    print(f"  Score: {res['best_score']:.4f}")
    print(f"  Steps: {res['refinement_steps']}")
    print(f"  Failure: {res['failure_mode']}")
    print(f"  Program: {res['best_program']}")

Loaded tasks: 1000
Converted tasks: 1000

TASK 0 | ID: 00576224
TEST: ARC Task 00576224
Best program:
rotate90 -> row_alt_tile_n(n=3) -> rotate270
Score: 0.8167
Refinement steps: 3
Error density progression: [0.3333, 0.1667]
Failure mode: near_miss_or_missing_operator

Per-train-pair evaluation:
  Pair 1:
    score = 0.7583
    pixel_accuracy = 0.6667
    structural_score = 0.8333
    color_score = 1.0000
    error_density = 0.3333
    error_count = 12
  Pair 2:
    score = 0.8750
    pixel_accuracy = 0.8333
    structural_score = 0.9000
    color_score = 1.0000
    error_density = 0.1667
    error_count = 6


TASK 1 | ID: 007bbfb7
TEST: ARC Task 007bbfb7
Best program:
rotate270 -> row_alt_tile_n(n=3) -> rotate90
Score: 0.8303
Refinement steps: 3
Error density progression: [0.2963, 0.2222, 0.2716, 0.2469, 0.1728]
Failure mode: near_miss_or_missing_operator

Per-train-pair evaluation:
  Pair 1:
    score = 0.7910
    pixel_accuracy = 0.7037
    structural_score = 0.8750
    color_score 

In [148]:
# ============================================
# AOFE OPS FULL REBUILD + PATCH
# ============================================

import numpy as np

# ----------------------------------
# CORE OPERATORS (FUNCTIONS)
# ----------------------------------

def rotate90(grid):
    return np.rot90(grid, k=1)

def rotate180(grid):
    return np.rot90(grid, k=2)

def rotate270(grid):
    return np.rot90(grid, k=3)

def reflect_h(grid):
    return np.flipud(grid)

def reflect_v(grid):
    return np.fliplr(grid)

def color_map(grid, mapping):
    grid = np.array(grid)
    out = grid.copy()
    for k, v in mapping.items():
        out[grid == k] = v
    return out

def recolor_role(grid, role, color, objects=None, roles=None):
    # minimal safe version (assumes roles already computed)
    grid = np.array(grid)
    out = grid.copy()

    if roles and role in roles:
        obj = roles[role]
        for (r, c) in obj["pixels"]:
            out[r, c] = color

    return out

def translate_role(grid, role, dx, dy, objects=None, roles=None):
    grid = np.array(grid)
    out = np.zeros_like(grid)

    if roles and role in roles:
        obj = roles[role]
        for (r, c) in obj["pixels"]:
            nr, nc = r + dy, c + dx
            if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]:
                out[nr, nc] = grid[r, c]

    return out

def row_alt_tile_n(grid, n):
    grid = np.array(grid)
    return np.tile(grid, (1, n))


# ----------------------------------
# 🔥 NEW OPERATORS
# ----------------------------------

def tile_full(grid, nx, ny):
    grid = np.array(grid)
    return np.tile(grid, (nx, ny))

def conditional_color_map(grid, source, target):
    grid = np.array(grid)
    out = grid.copy()
    out[grid == source] = target
    return out


# ----------------------------------
# BUILD OPS LIST
# ----------------------------------

ops = [

    # rotations
    ("rotate90", {}),
    ("rotate180", {}),
    ("rotate270", {}),

    # reflections
    ("reflect_h", {}),
    ("reflect_v", {}),

    # base tiling
    ("row_alt_tile_n", {"n": 2}),
    ("row_alt_tile_n", {"n": 3}),

    # object ops
    ("recolor_role", {"role": "largest", "color": 1}),
    ("translate_role", {"role": "largest", "dx": 1, "dy": 0}),

    # tile full (NEW)
    ("tile_full", {"nx": 2, "ny": 2}),
    ("tile_full", {"nx": 3, "ny": 3}),
]


# ----------------------------------
# ADD CONDITIONAL COLOR VARIANTS
# ----------------------------------

for s in range(10):
    for t in range(10):
        if s != t:
            ops.append(("conditional_color_map", {"source": s, "target": t}))


# ----------------------------------
# BASIC COLOR MAP (keep simple)
# ----------------------------------

ops.append(("color_map", {"mapping": {0: 0, 1: 2}}))


print("✅ OPS READY")
print("Total operators:", len(ops))

✅ OPS READY
Total operators: 102


In [149]:
print("ops exists:", "ops" in globals())
print(len(ops))

ops exists: True
102


In [150]:
print("ops exists:", "ops" in globals())

ops exists: True


In [151]:
# ============================================
# AOFE OPERATOR PATCH (TILE + CONDITIONAL COLOR)
# ============================================

import numpy as np

# ----------------------------------
# TILE OPERATOR
# ----------------------------------
def tile_full(grid, nx, ny):
    grid = np.array(grid)
    return np.tile(grid, (nx, ny))


# ----------------------------------
# CONDITIONAL COLOR MAP
# ----------------------------------
def conditional_color_map(grid, source, target):
    grid = np.array(grid)
    out = grid.copy()
    out[grid == source] = target
    return out


# ----------------------------------
# REGISTER OPERATORS INTO ops
# ----------------------------------

def register_new_ops(ops):

    op_names = set([op[0] for op in ops])

    # ---- TILE ----
    if "tile_full" not in op_names:
        ops.append(("tile_full", {"nx": 2, "ny": 2}))
        ops.append(("tile_full", {"nx": 3, "ny": 3}))

    # ---- CONDITIONAL COLOR MAP ----
    if "conditional_color_map" not in op_names:
        for s in range(10):
            for t in range(10):
                if s != t:
                    ops.append(("conditional_color_map", {"source": s, "target": t}))

    return ops


# ----------------------------------
# APPLY PATCH
# ----------------------------------

ops = register_new_ops(ops)

print("✅ Operators patched.")
print("Total operators:", len(ops))

✅ Operators patched.
Total operators: 102


In [153]:
# ============================================
# AOFE v13.5 BEAM SEARCH (FULL FIXED VERSION)
# ============================================

def beam_search(task, ops=None, width=10, depth=5):

    # ----------------------------------
    # AUTO-INJECT OPS (CRITICAL FIX)
    # ----------------------------------
    if ops is None:
        if "ops" not in globals():
            raise RuntimeError("ops not defined globally")
        ops = globals()["ops"]

    best_program = None
    best_score = -1
    best_analysis = None
    error_density_progression = []

    # ----------------------------------
    # helper: evaluate candidate
    # ----------------------------------
    def consider(program, refinement_steps):

        nonlocal best_score, best_program, best_analysis

        result = eval_program_with_errors(program, task)

        if result is None:
            return None

        score = result["score"]
        analyses = result["analyses"]

        # MDL bias → shorter programs preferred
        mdl_penalty = len(program) * 0.001
        adjusted_score = score - mdl_penalty

        if adjusted_score > best_score:
            best_score = adjusted_score
            best_program = program
            best_analysis = result

        return (adjusted_score, -len(program), program, refinement_steps, analyses, result)

    # ============================================
    # INITIAL CANDIDATE GENERATION (FIXED)
    # ============================================

    initial_programs = [[]]

    for op in ops:
        initial_programs.append([op])

    ranked = []

    for prog in initial_programs:
        item = consider(prog, refinement_steps=0)
        if item is not None:
            ranked.append(item)

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)
    ranked = ranked[:width]

    # ============================================
    # MAIN SEARCH LOOP
    # ============================================

    for step in range(depth):

        candidates = []

        for _, _, program, _, analyses, _ in ranked:

            if not analyses:
                continue

            worst_analysis = max(analyses, key=lambda a: a["error_density"])
            error_density_progression.append(worst_analysis["error_density"])

            ordered_ops = get_priority_ops(worst_analysis, ops)

            last_op = program[-1][0] if program else None

            # PRIMARY: avoid repeating same op
            for op in ordered_ops[:width]:

                if op[0] == last_op:
                    continue

                new_program = program + [op]
                item = consider(new_program, refinement_steps=step + 1)

                if item is not None:
                    candidates.append(item)

            # SECONDARY: allow limited repetition
            for op in ordered_ops[:2]:

                if op[0] == last_op:
                    new_program = program + [op]
                    item = consider(new_program, refinement_steps=step + 1)

                    if item is not None:
                        candidates.append(item)

        if not candidates:
            break

        candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
        ranked = candidates[:width]

        if best_score >= 1.0:
            break

    # ============================================
    # RETURN RESULT
    # ============================================

    return {
        "best_program": best_program,
        "best_score": best_score,
        "refinement_steps": len(error_density_progression),
        "error_density_progression": error_density_progression,
        "failure_mode": None if best_score >= 0.999 else "near_miss_or_missing_operator"
    }

In [156]:
# ============================================
# FIXED: CONVERT TO SOLVER FORMAT (ARC CORRECT)
# ============================================

def convert_to_solver_format(tasks):

    solver_tasks = {}

    for task_id, task_data in tasks.items():

        # already correct format
        if isinstance(task_data, dict) and "train" in task_data:
            solver_tasks[task_id] = task_data
            continue

        # fallback: convert list → ARC format
        if isinstance(task_data, list):

            train_pairs = []

            for pair in task_data:
                if "input" in pair and "output" in pair:
                    train_pairs.append({
                        "input": pair["input"],
                        "output": pair["output"]
                    })

            solver_tasks[task_id] = {
                "train": train_pairs,
                "test": []
            }

        else:
            raise ValueError(f"Unknown task format for {task_id}")

    return solver_tasks

In [158]:
# ============================================
# 🔥 FORCE RELOAD + RECONVERT (HARD RESET FIX)
# ============================================

import json

# ----------------------------------
# RELOAD RAW DATA (IMPORTANT)
# ----------------------------------
DATA_PATH = "/content/arc-agi_training_challenges.json"

tasks = json.load(open(DATA_PATH))

print("Loaded raw tasks:", len(tasks))


# ----------------------------------
# RECONVERT USING FIXED FUNCTION
# ----------------------------------
solver_tasks = convert_to_solver_format(tasks)

print("Converted tasks:", len(solver_tasks))


# ----------------------------------
# 🔍 VERIFY FORMAT (CRITICAL)
# ----------------------------------
sample_key = list(solver_tasks.keys())[0]
sample_task = solver_tasks[sample_key]

print("\nSample task ID:", sample_key)
print("Type:", type(sample_task))

if isinstance(sample_task, dict):
    print("Keys:", sample_task.keys())
    print("Train count:", len(sample_task["train"]))
else:
    print("❌ STILL WRONG FORMAT")

# HARD ASSERT (WILL CRASH IF WRONG — GOOD)
assert isinstance(sample_task, dict)
assert "train" in sample_task

print("\n✅ FORMAT VERIFIED — READY FOR RUN")

Loaded raw tasks: 1000
Converted tasks: 1000

Sample task ID: 00576224
Type: <class 'dict'>
Keys: dict_keys(['train', 'test'])
Train count: 2

✅ FORMAT VERIFIED — READY FOR RUN


In [162]:
# ============================================
# FIXED: RUN CONTROLLED TASK (SAFE + CORRECT)
# ============================================

def run_controlled_task(name, task, width=10, depth=5):

    print("="*70)
    print(f"TEST: {name}")
    print("="*70)

    try:
        # ----------------------------------
        # 🔥 ENSURE CORRECT FORMAT
        # ----------------------------------
        if isinstance(task, list):
            raise ValueError("❌ Task is still a list — conversion failed")

        if "train" not in task:
            raise ValueError("❌ Task missing 'train' key")

        # ----------------------------------
        # RUN BEAM SEARCH
        # ----------------------------------
        result = beam_search(task, width=width, depth=depth)

        print("\nBest program:")
        print(result["best_program"])
        print(f"Score: {result['best_score']:.4f}")
        print(f"Refinement steps: {result['refinement_steps']}")
        print(f"Error density progression: {result['error_density_progression']}")
        print(f"Failure mode: {result['failure_mode']}")

        return result

    except Exception as e:
        print("ERROR during test:")
        import traceback
        traceback.print_exc()

        return {
            "best_program": None,
            "best_score": 0.0,
            "refinement_steps": 0,
            "error_density_progression": [],
            "failure_mode": "crash"
        }

In [164]:
# ============================================
# FIXED: eval_program_with_errors (HARD SAFE)
# ============================================

def eval_program_with_errors(program, task):

    import numpy as np

    # ----------------------------------
    # 🔥 FORCE CORRECT FORMAT
    # ----------------------------------
    if isinstance(task, list):
        raise ValueError("❌ eval_program received LIST instead of dict")

    if "train" not in task:
        raise ValueError("❌ task missing 'train' key")

    train_pairs = task["train"]

    total_score = 0
    analyses = []

    for pair in train_pairs:

        inp = np.array(pair["input"])
        expected = np.array(pair["output"])

        try:
            output = run_program(program, inp)
        except Exception:
            return None

        if output is None:
            return None

        output = np.array(output)

        # ----------------------------------
        # SHAPE CHECK
        # ----------------------------------
        if output.shape != expected.shape:
            pixel_accuracy = 0.0
            error_map = np.ones_like(expected)
        else:
            error_map = (output != expected).astype(int)
            pixel_accuracy = 1.0 - np.mean(error_map)

        error_count = int(np.sum(error_map))
        error_density = error_count / error_map.size

        # simple structural + color scores
        structural_score = 1.0 if output.shape == expected.shape else 0.0
        color_score = len(np.unique(output)) / max(1, len(np.unique(expected)))

        score = (
            0.5 * pixel_accuracy +
            0.25 * structural_score +
            0.25 * color_score
        )

        total_score += score

        analyses.append({
            "pixel_accuracy": pixel_accuracy,
            "structural_score": structural_score,
            "color_score": color_score,
            "error_density": error_density,
            "error_count": error_count
        })

    avg_score = total_score / len(train_pairs)

    return {
        "score": avg_score,
        "analyses": analyses
    }

In [168]:
# ============================================
# AOFE OPERATOR UPGRADE (CRITICAL MISSING OPS)
# ============================================

import numpy as np


# ----------------------------------
# CROP TO NON-ZERO BOUNDING BOX
# ----------------------------------
def crop_to_bbox(grid):
    grid = np.array(grid)

    rows = np.any(grid != 0, axis=1)
    cols = np.any(grid != 0, axis=0)

    if not np.any(rows) or not np.any(cols):
        return grid

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    return grid[rmin:rmax+1, cmin:cmax+1]


# ----------------------------------
# EXTRACT LARGEST OBJECT ONLY
# ----------------------------------
def extract_largest_object(grid):
    grid = np.array(grid)

    objects = extract_objects(grid)

    if not objects:
        return grid

    largest = max(objects, key=lambda o: o["area"])

    out = np.zeros_like(grid)

    for (r, c) in largest["pixels"]:
        out[r, c] = grid[r, c]

    return out


# ----------------------------------
# REGISTER NEW OPS
# ----------------------------------

def add_critical_ops(ops):

    names = set(op[0] for op in ops)

    if "crop_to_bbox" not in names:
        ops.append(("crop_to_bbox", {}))

    if "extract_largest_object" not in names:
        ops.append(("extract_largest_object", {}))

    return ops


# APPLY
ops = add_critical_ops(ops)

print("✅ Critical operators added")
print("Total ops:", len(ops))

✅ Critical operators added
Total ops: 104


In [169]:
# ============================================
# FIXED: run_program (ROBUST EXECUTION)
# ============================================

def run_program(program, grid):

    import numpy as np

    current = np.array(grid)

    for op_name, params in program:

        try:
            # ----------------------------------
            # SIMPLE OPS (no extra args)
            # ----------------------------------
            if op_name in ["rotate90", "rotate180", "rotate270", "reflect_h", "reflect_v"]:
                current = globals()[op_name](current)

            # ----------------------------------
            # COLOR MAP
            # ----------------------------------
            elif op_name == "color_map":
                current = color_map(current, params["mapping"])

            elif op_name == "conditional_color_map":
                current = conditional_color_map(current, params["source"], params["target"])

            # ----------------------------------
            # TILE
            # ----------------------------------
            elif op_name == "tile_full":
                current = tile_full(current, params["nx"], params["ny"])

            elif op_name == "row_alt_tile_n":
                current = row_alt_tile_n(current, params["n"])

            # ----------------------------------
            # OBJECT-BASED OPS
            # ----------------------------------
            elif op_name in ["recolor_role", "translate_role"]:

                objects = extract_objects(current)
                roles = infer_object_roles_v2(objects, current)

                if op_name == "recolor_role":
                    current = recolor_role(
                        current,
                        params["role"],
                        params["color"],
                        objects=objects,
                        roles=roles
                    )

                elif op_name == "translate_role":
                    current = translate_role(
                        current,
                        params["role"],
                        params["dx"],
                        params["dy"],
                        objects=objects,
                        roles=roles
                    )

            # ----------------------------------
            # UNKNOWN OP (SAFE FAIL)
            # ----------------------------------
            else:
                return None

        except Exception:
            return None

    return current

In [170]:
# ============================================
# RUN 5 ARC TASKS (FINAL CHECK)
# ============================================

results = []

for i, (task_id, task) in enumerate(solver_tasks.items()):
    if i >= 5:
        break

    print("\n" + "="*70)
    print(f"TASK {i} | ID: {task_id}")
    print("="*70)

    result = run_controlled_task(
        f"ARC Task {task_id}",
        task,
        width=15,
        depth=5
    )

    results.append((task_id, result))


# ----------------------------------
# SUMMARY
# ----------------------------------
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

for task_id, res in results:
    print(f"\nTask: {task_id}")
    print(f"  Score: {res['best_score']:.4f}")
    print(f"  Steps: {res['refinement_steps']}")
    print(f"  Failure: {res['failure_mode']}")
    print(f"  Program: {res['best_program']}")


TASK 0 | ID: 00576224
TEST: ARC Task 00576224

Best program:
[('tile_full', {'nx': 3, 'ny': 3}), ('recolor_role', {'role': 'largest', 'color': 1})]
Score: 0.8904
Refinement steps: 75
Error density progression: [0.3333333333333333, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.3611111111111111, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.5, 0.6666666666666666, 0.6666666666666666, 0.8333333333333334, 0.8333333333333334, 1.0, 1.0, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3611111111111111, 0.3